# Variant Calling, PCA and Association Mapping of *Hypolimnas misippus* Wing Morphs with ANGSD, PLINK and SNPTEST

**Course:** Genome-Wide Association Studies (GWAS) for Evolutionary Biology
**Instructor:** Nikolay Oskolkov
**Platform:** Google Colab
**Organism:** *Hypolimnas misippus* (the Diadem or Danaid eggfly), a pantropical butterfly
**Data and biology from:** Orteu et al. (2024) *Transposable element insertions are associated with Batesian mimicry in the pantropical butterfly Hypolimnas misippus.* Molecular Biology and Evolution 41(3): msae041.

---

This notebook is the **continuation** of `QC_Alignment_Hypolimnas_misippus_Colab.ipynb`. That notebook ended with one **sorted, indexed BAM file per butterfly** (folder `bams/`) and a table `phenotypes.tsv` that links every BAM to its wing morph. Here we start from exactly that folder and go all the way to **association results from three different programs**.

We will:

1. Install **Conda** (via `condacolab`) and the tools **ANGSD, pcangsd, PLINK, SAMtools, BCFtools/HTSlib and SNPTEST**.
2. Mount **Google Drive**, find the `bams/` folder produced by the QC notebook, and check that every BAM is healthy and carries the right sample name.
3. Learn, with a small simulation, **why genotypes cannot simply be "called" at low coverage**, and what *genotype likelihoods* are.
4. Run a **depth pilot** with ANGSD to choose sensible site filters.
5. Do **probabilistic variant calling with ANGSD** (genotype likelihoods, SNP calling, genotype posteriors) and write the results as BEAGLE, MAF and BCF/VCF files.
6. Run **PCA with pcangsd** to look at population structure (the admixture analysis of the ANGSD lab is deliberately skipped).
7. Test every variant for association with the wing morph (**`m` versus `i`**) with three programs that treat genotype uncertainty in three different ways: **ANGSD `-doAsso`**, **PLINK** and **SNPTEST**.
8. Write **three VCF files**, one per program, each containing the variants that program actually tested, and **compare them**: how many variants, where in the genome, which alleles, which p-values.

> **How to use this notebook.** Run the cells from top to bottom. The first code cell restarts the Colab runtime on purpose (Conda installation). All long steps are **resumable**: they skip outputs that already exist on Drive. To force a step to run again, delete its output files (or set the `FORCE` flag in Section 3.1).
>
> **Expect to read as well as run.** Every section starts with the *idea*, then the *command*, then a cell that *looks inside the output*. The numbers you obtain depend on how much sequence is in your small FASTQ subsets, so we always print the numbers instead of quoting them.

## 0. Background: what are we about to do, and why is it harder than it looks?

### 0.1 Where the previous notebook left us

| File / folder | Created in the QC notebook | Used here for |
|---|---|---|
| `bams/<sampleID>.sorted.bam` (+ `.bai`) | Section 10 (BWA-MEM + SAMtools) | **all** analyses |
| `phenotypes.tsv` | Section 13 | sample IDs, morph, binary phenotype (`m` = 1, `i` = 0), mean depth |
| `alignment_summary.tsv` | Section 11 | per-sample depth, to look for technical confounding |
| `reference_HypMis_v2/` | Section 9 | chromosome names and lengths; identification of chromosome 26 (M locus), Z and W |

Our question is the one the authors of the paper asked: **which genetic variants distinguish the two wing morphs?**

* **`m`** = *misippus* morph, genotype **`M-`** (at least one copy of the dominant allele **M**). Coded **1** ("case").
* **`i`** = *inaria / immima* morph, genotype **`mm`**. Coded **0** ("control").

The morph is controlled mainly by one locus of large effect (**M**) on **chromosome 26**, in a ~12 kb window at 6,731,000 to 6,743,400 bp. If the pipeline works, the strongest association signal should come from there. This is our **positive control**, just as the read-depth plot was in the QC notebook.

### 0.2 The central difficulty: genotypes are *uncertain* at low coverage

A standard GWAS (PLINK, SNPTEST, ...) assumes that every individual has a **known genotype** at every variant: `0`, `1` or `2` copies of an allele. With sequencing data at about **1x or less** that assumption fails:

* at a typical site an individual has **0 or 1 reads**, sometimes 2 or 3;
* with one read showing allele `A` you cannot tell whether the individual is `AA` (the read was sampled from either chromosome) or `AB` (you happened to sample the `A` chromosome);
* sequencing errors add further uncertainty.

So the honest statement is *not* "this individual is `AB`" but "the data make `AA`, `AB` and `BB` this likely". That statement is a **genotype likelihood (GL)**.

**Definition.** For an individual with sequencing data $D$ at a biallelic site, and for each of the three possible genotypes $g \in \{0,1,2\}$ (number of copies of the *minor* allele),

$$\mathrm{GL}(g) = P(D \mid G=g) = \prod_{\text{reads } r} P(b_r \mid g), \qquad
P(b_r = \text{minor} \mid g) = \tfrac{g}{2}(1-\varepsilon_r) + \bigl(1-\tfrac{g}{2}\bigr)\varepsilon_r ,$$

where $b_r$ is the base observed in read $r$ and $\varepsilon_r = 10^{-Q_r/10}$ is the probability that the base is a sequencing error, derived from its Phred quality $Q_r$. (This is the idea behind all GL models; the GATK model used by ANGSD `-GL 2` and the SAMtools model `-GL 1` add refinements such as dependence between errors.) Reads are assumed independent given the genotype.

The GLs are **not probabilities of genotypes**. To obtain those we need a **prior** (Bayes' rule). Assuming Hardy-Weinberg equilibrium and an allele frequency $f$ of the minor allele, the prior is $P(g) = (1-f)^2,\ 2f(1-f),\ f^2$ for $g = 0, 1, 2$, and the **genotype posterior** is

$$P(G=g \mid D) = \frac{P(D \mid G=g)\,P(g)}{\sum_{g'} P(D \mid G=g')\,P(g')} .$$

ANGSD estimates $f$ from the data of *all* individuals (`-doMaf 1`) and then computes these posteriors (`-doPost 1`). The posterior also gives the **expected genotype (dosage)**, $E[G \mid D] = P(G{=}1\mid D) + 2\,P(G{=}2\mid D)$, a number between 0 and 2.

There are then three ways to feed such data into an association test, and they correspond to the three programs of this notebook:

| Strategy | What the test sees | Program here | Price / advantage |
|---|---|---|---|
| **Hard calls** | Pick the most probable genotype, set it to *missing* if the posterior is below a cut-off (for example 0.9) | **PLINK** | Fast, standard, thousands of tools. But at low coverage *most* genotypes become missing and the surviving calls are biased towards homozygotes and towards well-covered regions. |
| **Genotype probabilities / dosages** | The three posterior probabilities of every individual, used as weights or turned into a dosage | **SNPTEST** | Keeps all individuals; uncertainty is propagated into the test. |
| **Genotype likelihoods / posteriors inside the test** | Likelihoods or posteriors used directly in a score test, a likelihood-ratio test, or an EM "latent genotype" model | **ANGSD `-doAsso`** | Designed for low coverage; no information is thrown away. |

### 0.3 What ANGSD does, step by step

ANGSD (Analysis of Next Generation Sequencing Data; Korneliussen et al. 2014) never calls genotypes unless you ask. For every site that passes the read filters it:

1. **Counts bases** and computes **genotype likelihoods** (`-GL`).
2. Infers the **major and minor allele** (`-doMajorMinor 1`: from the likelihoods; `4`: major = reference).
3. Estimates the **minor-allele frequency** by maximum likelihood with an EM algorithm that uses all individuals together (`-doMaf 1`).
4. Tests whether the site is **polymorphic** with a likelihood-ratio test of "frequency = 0" versus "frequency estimated" (`-SNP_pval`). This is **SNP calling in the probabilistic sense**: a SNP is a site where the data reject monomorphism, not a site where someone saw two alleles.
5. Computes the **genotype posteriors** (`-doPost 1`, frequency as prior), and writes everything to files: BEAGLE (`-doGlf 2`), MAF table, BCF (`-doBcf 1`), association results (`-doAsso`).

### 0.4 The association tests in one paragraph each

*Allelic / trend tests with hard calls (PLINK `--assoc`).* Count minor and major alleles in cases and controls and compare the two proportions with a 2x2 chi-square test (1 degree of freedom). Simple and robust, but it treats the 2n alleles as independent and requires known genotypes.

*Score test on posteriors (ANGSD `-doAsso 2`).* Fit the *null* logistic model (phenotype $\sim$ intercept + covariates), then ask how strongly the expected genotype (dosage) is correlated with the residuals. Because only the null model is fitted, it is very fast. It is a good test when genotypes are uncertain but the effect is moderate. Mode **1** is a likelihood-ratio test of allele frequency differences between cases and controls computed directly from the genotype likelihoods. Mode **4** is the **latent genotype model**: the true genotype is treated as unobserved, and the regression is fitted by EM. It also gives an effect size (`beta`) and its standard error.

*SNPTEST with `-method score`.* A missing-data score test: genotypes are known only through their probabilities, and the likelihood sums over the uncertainty. It reports a p-value, an **information measure** (`info`, between 0 and 1: how well the genotypes are determined; 1 = perfectly) and the allele frequencies.

All three produce a **p-value per variant** for the null hypothesis "this variant is not associated with the morph".

### 0.5 Biology of the expected result (so we know what "correct" looks like)

* The phenotype is controlled by a **dominant** allele: `M-` gives *misippus*, `mm` gives *inaria*. An *additive* test (one degree of freedom, "more copies, more effect") is therefore not the perfect model, but it still has good power because the association is very strong.
* The reference genome was assembled from an **M homozygote**. The dominant allele carries two **transposable-element insertions** (2.4 kb and 4.3 kb) that the recessive allele lacks. Reads from `mm` individuals cannot map inside the insertions. In the SNP data this means *missingness that depends on the phenotype*: no reads from `i` samples over the insertions. Hard-call programs (PLINK) see this as **differential missingness**; we will test for it explicitly.
* All butterflies are **females** (ZW). The Z chromosome therefore has one copy per individual and the W is female-specific. A diploid genotype model is wrong there, so we analyse **autosomes only** by default.

### 0.6 Population structure, PCA and genomic inflation

If the sample consists of genetically distinct groups that also differ in the phenotype, *every* variant that differs between the groups will show association: false positives caused by ancestry. The classical remedy is to compute the **principal components (PCs)** of the genome-wide genotype covariance matrix, look at them, and include the top PCs as **covariates**. For low-coverage data this is done on genotype likelihoods: **pcangsd** (Meisner & Albrechtsen 2018) estimates the individual-by-individual covariance matrix from the BEAGLE file, and we take its eigenvectors.

We then judge the result with:

* the **QQ plot** (observed versus expected $-\log_{10}P$): under the null the points follow the diagonal; a systematic lift of the whole cloud means inflation, a few points leaving the line at the top means true signals;
* the **genomic inflation factor** $\lambda_{GC}$ = median($\chi^2_{obs}$) / 0.4549; values near 1 are good, values above ~1.1 suggest stratification or technical artefacts.

### 0.7 Multiple testing

We test tens of thousands of variants. At $\alpha = 0.05$ we would expect 5% false positives, so we use the **Bonferroni threshold** $0.05 / n_{tests}$. Notice what this implies for our comparison: **a program that tests fewer variants has a less strict threshold**. The number of variants is not only a bookkeeping detail; it changes the conclusions.

### 0.8 Map of the notebook

| Section | Tool | Input | Output |
|---|---|---|---|
| 1-2 | `condacolab`, `mamba` | | ANGSD, pcangsd, PLINK, SAMtools, BCFtools, SNPTEST |
| 3 | Python | `bams/`, `phenotypes.tsv` | checked sample table, BAM list, phenotype files |
| 4 | Python | | toy genotype-likelihood demo |
| 5 | SAMtools | BAM header / reference | chromosome table, regions file (autosomes) |
| 6 | ANGSD `-doDepth` | BAM | depth per sample, filters `-minInd`, `-setMaxDepth` |
| 7 | ANGSD | BAM | `.beagle.gz`, `.mafs.gz`, `.bcf` |
| 8 | pcangsd | `.beagle.gz` | `.cov`, PCs, plots |
| 9 | ANGSD `-doAsso` | BAM + phenotype | `.lrt0.gz` |
| 10 | PLINK | hard-called VCF | `.assoc`, filtered genotypes |
| 11 | SNPTEST | `.gen` + `.sample` | association table |
| 12 | BCFtools | all of the above | **3 VCF files** |
| 13 | Python | 3 VCFs | comparison of counts, positions, alleles, p-values |

### 0.9 What exactly are "the three VCF files"?

Every association program reads genotype information in its own way and applies its own quality filters. After the analyses we therefore write, for each program, **one VCF containing exactly the variants it tested**, with its own test statistic stored in the `INFO` column:

| VCF | Variants it contains | Genotype information inside | Filters that shape it |
|---|---|---|---|
| **VCF 1: ANGSD** | sites for which `angsd -doAsso 2` returned a test | genotype posteriors (`GP`) and likelihoods (`GL`) | read filters, `-minInd`, `-SNP_pval`, `-minMaf`, `-setMaxDepth`, `-minHigh`, `-minCount` |
| **VCF 2: PLINK** | variants that survived hard-call quality control and were tested | **hard calls** (`GT`), many missing | posterior cut-off, `--geno`, `--maf`, `--hwe` |
| **VCF 3: SNPTEST** | variants with a valid SNPTEST test and sufficient `info` | genotype posteriors (`GP`) | SNPTEST failures, information threshold |

All three start from the **same ANGSD variant calls** (the "universe" of Section 7), so every difference between them is caused by *how the program represents genotype uncertainty and which filters it applies*. That is exactly what we want students to see.

## 1. Install Conda in Google Colab

Google Colab does not ship with Conda. The `condacolab` package installs a lightweight **Miniforge** (Conda plus the fast `mamba` solver) directly into the running notebook.

> ⚠️ **Important:** After this cell finishes, the Colab kernel **restarts automatically**. This is expected. Once the restart is done, simply continue with the next cell. You do **not** need to re-run this cell.

In [ ]:
# Install Conda (Miniforge) inside Google Colab. The kernel restarts automatically afterwards.
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. Install ANGSD, pcangsd, PLINK, SAMtools, BCFtools and SNPTEST

| Tool | What we use it for | Source |
|---|---|---|
| **ANGSD** | genotype likelihoods, SNP calling, genotype posteriors, `-doAsso` association | Bioconda |
| **pcangsd** | PCA from genotype likelihoods | Bioconda |
| **PLINK 1.9** | association with hard-called genotypes | Bioconda (`plink`) |
| **SAMtools** | read BAM headers, check BAM integrity | Bioconda |
| **BCFtools + HTSlib** (`bgzip`, `tabix`) | convert, filter, annotate and index VCF/BCF files | Bioconda |
| **SNPTEST v2.5.6** | association with genotype probabilities | channel `dnachun` (a community Conda build of the Oxford program) |

SNPTEST is *not* in Bioconda. Its authors distribute it from the Oxford website (<https://www.chg.ox.ac.uk/~gav/snptest/>); we try the `dnachun` Conda build first. If that fails, the notebook prints instructions for a manual download. Installation takes a few minutes.

In [ ]:
%%bash
# Main toolchain from Bioconda (+ conda-forge for the dependencies).
mamba install -y -q -c conda-forge -c bioconda angsd samtools bcftools htslib plink 2>&1 | tail -n 5

In [ ]:
%%bash
# pcangsd in a separate command: if the Conda build conflicts with the other tools, fall back to pip.
mamba install -y -q -c conda-forge -c bioconda pcangsd 2>&1 | tail -n 3 || true
if ! command -v pcangsd >/dev/null 2>&1; then
    echo "pcangsd not found after the Conda install; trying pip ..."
    pip install -q pcangsd
fi
command -v pcangsd

In [ ]:
%%bash
# SNPTEST from the dnachun channel; fall back to a direct download of an older static build.
if ! command -v snptest >/dev/null 2>&1; then
    mamba install -y -q -c conda-forge -c dnachun snptest 2>&1 | tail -n 3 || true
fi
if ! command -v snptest >/dev/null 2>&1; then
    echo "Conda build not available; trying the static build of SNPTEST v2.5.4-beta3 from Oxford ..."
    mkdir -p /content/snptest_dl && cd /content/snptest_dl
    for base in https://www.chg.ox.ac.uk/~gav/resources https://www.well.ox.ac.uk/~gav/resources; do
        wget -q "$base/snptest_v2.5.4-beta3_linux_x86_64_static.tgz" -O snptest.tgz && break || true
    done
    if [ -s snptest.tgz ] && tar -xzf snptest.tgz 2>/dev/null; then
        bin=$(find . -type f -name "snptest_v*" | head -n 1)
        [ -n "$bin" ] && cp "$bin" /usr/local/bin/snptest && chmod +x /usr/local/bin/snptest
    fi
fi
if command -v snptest >/dev/null 2>&1; then
    echo "SNPTEST found at: $(command -v snptest)"
else
    echo "SNPTEST could NOT be installed automatically."
    echo "Download it from https://www.chg.ox.ac.uk/~gav/snptest/ , upload the executable to your Drive,"
    echo "and set SNPTEST_BINARY in Section 3.1 to its path."
fi

### 2.1 Verify the installation

Recording the **exact version** of every program is part of reproducible science: the versions belong in the Methods section of any paper that uses this pipeline.

In [ ]:
%%bash
echo "ANGSD:    $(angsd 2>&1 | grep -i -m1 'version')"
echo "pcangsd:  $(pcangsd --help 2>&1 | head -n 1)"
echo "PLINK:    $(plink --version 2>&1 | head -n 1)"
echo "SAMtools: $(samtools --version | head -n 1)"
echo "BCFtools: $(bcftools --version | head -n 1)"
echo "bgzip:    $(bgzip --version | head -n 1)"
echo "tabix:    $(tabix --version 2>&1 | head -n 1)"
echo "SNPTEST:  $(snptest 2>&1 | grep -i -m1 'snptest v' || echo 'not found (see Section 3.1: SNPTEST_BINARY)')"
echo
echo "CPU cores available: $(nproc)"
echo "Free memory:"; free -h | head -n 2
echo "Free disk on /content:"; df -h /content | tail -n 1

## 3. Mount Google Drive and Locate the BAM Files

Colab's local disk is wiped when the session ends, but **Google Drive persists**. The BAM files made by the QC notebook are on Drive, and all results of this notebook will be written there too (folder `association_HypMis/` next to `bams/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3.1 Locate the project and define all parameters

The next cell

* searches your Drive (up to six folder levels deep) for a folder called **`bams`** that contains `*.sorted.bam` files, and uses its **parent** as the *project folder* (the same folder that holds `phenotypes.tsv`, `reference_HypMis_v2/`, ...);
* creates the output folders under `association_HypMis/`;
* defines **all analysis parameters in one place**. Each parameter is explained in the section where it is used. Change them here, then re-run from this cell.

If the automatic search picks the wrong folder, set `BAM_DIR_OVERRIDE` (an example is given in the comment).

In [ ]:
import os, re, json, gzip, glob, shutil, subprocess, io, time, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---------------------------------------------------------------- where are the BAM files?
DRIVE_ROOT = Path("/content/drive/MyDrive")
BAM_FOLDER_NAME = "bams"
# If the automatic search fails or picks the wrong folder, set the path here, e.g.
# BAM_DIR_OVERRIDE = "/content/drive/MyDrive/GWAS_course/bams"
BAM_DIR_OVERRIDE = None

def find_on_drive(name, kind, maxdepth=6):
    # Search Google Drive for a folder (kind='d') or file (kind='f') with an exact name.
    res = subprocess.run(["find", str(DRIVE_ROOT), "-maxdepth", str(maxdepth),
                          "-type", kind, "-name", name], capture_output=True, text=True)
    return [Path(p) for p in res.stdout.splitlines() if p.strip()]

if BAM_DIR_OVERRIDE:
    BAM_DIR = Path(BAM_DIR_OVERRIDE)
else:
    hits = [p for p in find_on_drive(BAM_FOLDER_NAME, "d") if any(p.glob("*.sorted.bam"))]
    assert hits, (f"Could not find a folder called '{BAM_FOLDER_NAME}' with *.sorted.bam files in your Google Drive. "
                  "Run the QC notebook first, or set BAM_DIR_OVERRIDE above.")
    hits.sort(key=lambda p: (not (p.parent / "phenotypes.tsv").exists(), str(p)))   # prefer the one with phenotypes.tsv next to it
    if len(hits) > 1:
        print("WARNING: several matching folders found, using the first one:")
        for h in hits: print("   ", h)
    BAM_DIR = hits[0]
assert BAM_DIR.is_dir(), f"{BAM_DIR} is not a directory"
PROJECT_DIR = BAM_DIR.parent

def locate_file(name):
    p = PROJECT_DIR / name
    if p.exists():
        return p
    hits = find_on_drive(name, "f")
    return hits[0] if hits else None

# ---------------------------------------------------------------- output folders (on Drive = persistent)
OUT = PROJECT_DIR / "association_HypMis"
D = {k: OUT / k for k in ["angsd", "pcangsd", "plink", "snptest", "vcf", "figures"]}
WORK = Path("/content/work_assoc")            # local disk: fast but temporary
LOCAL_BAMS = WORK / "bams"
for d in list(D.values()) + [WORK]:
    d.mkdir(parents=True, exist_ok=True)

# ================================================================ ANALYSIS PARAMETERS
THREADS = os.cpu_count() or 2
FORCE = False                 # True = redo every step even if its output exists
COPY_BAMS_LOCAL = True        # copy BAMs from Drive to the local disk first (much faster for ANGSD; small BAMs only)

# ---- read and site filters for ANGSD (explained in Sections 6 and 7)
MIN_MAPQ = 30                 # discard reads with mapping quality below this
MIN_BASEQ = 20                # discard bases with Phred quality below this
GL_MODEL = 2                  # 1 = SAMtools GL, 2 = GATK GL
SNP_PVAL = 1e-6               # keep a site as a SNP only if the likelihood-ratio test of monomorphism has P below this
MIN_MAF = 0.05                # minimum minor allele frequency
MIN_IND = None                # minimum number of individuals with data at a site. None = chosen from the depth pilot (Section 6)
MAX_DEPTH = None              # maximum total depth over all individuals at a site. None = chosen from the depth pilot

# ---- phenotype and covariates
CASE_MORPH = "m"              # this morph is coded 1 ("case"), the other morph 0 ("control"), as in the paper
N_PCS_COV = 0                 # number of principal components used as covariates in ALL three tests (0 = none; see Section 8)

# ---- ANGSD association (Section 9)
ASSO_MODES = [1, 2, 4]        # 1 = allele-frequency LRT, 2 = score test, 4 = latent genotype model (EM)
ASSO_PRIMARY = 2              # the mode used for VCF 1 and for the comparison
ASSO_MIN_HIGH = 0             # ANGSD -minHigh: min. individuals with posterior > 0.9 in two genotype classes (0 = off; low-coverage data!)
ASSO_MIN_COUNT = 10           # ANGSD -minCount: min. expected minor-allele count (MAF x 2N)

# ---- PLINK (Section 10): hard calls
POST_CUTOFF = 0.9             # a genotype is called only if its posterior probability is at least this
PLINK_GENO = 0.95             # drop variants with more than this fraction of missing hard calls (0.95 = keep variants called in >= 5 % of individuals)
PLINK_MAF = 0.05              # drop variants with MAF (among called genotypes) below this
PLINK_HWE = 1e-6              # drop variants out of Hardy-Weinberg equilibrium (controls only) at this P

# ---- SNPTEST (Section 11)
SNPTEST_METHOD = "score"      # score | newml | threshold | expected
SNPTEST_MIN_INFO = 0.1        # keep variants whose SNPTEST 'info' is at least this for VCF 3
SNPTEST_BINARY = None         # e.g. "/content/drive/MyDrive/tools/snptest_v2.5.4-beta3" if the automatic installation failed

# ---- the M locus (positive control)
PEAK_START, PEAK_END = 6_731_000, 6_743_400
INCLUDE_Z = False             # females are ZW: the Z has one copy per individual, so we analyse autosomes only

# ---- colours
MORPH_ORDER = ["m", "i"]
MORPH_COLORS = {"m": "#4c72b0", "i": "#dd8452"}
TOOL_COLORS = {"ANGSD": "#0072B2", "PLINK": "#D55E00", "SNPTEST": "#009E73"}

# ---------------------------------------------------------------- make paths visible to %%bash cells
for k, v in {"PROJECT_DIR": PROJECT_DIR, "BAM_DIR": BAM_DIR, "OUT": OUT, "WORK": WORK, "THREADS": THREADS}.items():
    os.environ[k] = str(v)

if SNPTEST_BINARY:
    os.makedirs("/usr/local/bin", exist_ok=True)
    shutil.copy(SNPTEST_BINARY, "/usr/local/bin/snptest"); os.chmod("/usr/local/bin/snptest", 0o755)

print("Project folder :", PROJECT_DIR)
print("BAM folder     :", BAM_DIR, f"({len(list(BAM_DIR.glob('*.sorted.bam')))} BAM files)")
print("Output folder  :", OUT)
print("CPU threads    :", THREADS)
print("SNPTEST        :", shutil.which("snptest") or "NOT FOUND (see Section 2 / SNPTEST_BINARY)")

In [ ]:
# A small helper that runs a shell command, keeps the log on Drive and shows the end of the log.
def run(cmd, log=None, check=True, show=12, quiet=False):
    if not quiet:
        print("$", cmd if len(cmd) < 600 else cmd[:600] + " ...")
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    text = (r.stdout or "") + (r.stderr or "")
    if log is not None:
        Path(log).write_text(f"$ {cmd}\n\n{text}")
    if r.returncode != 0 and check:
        print(text[-3500:])
        raise RuntimeError(f"Command failed with exit code {r.returncode} (see the output above).")
    if show and text.strip() and not quiet:
        lines = text.strip().splitlines()
        print("\n".join(lines[-show:]))
    if not quiet:
        print(f"[done in {time.time() - t0:.1f} s, exit code {r.returncode}]")
    return r

def done(*paths):
    # True if every file exists and is non-empty (and FORCE is off): used to make steps resumable.
    return (not FORCE) and all(Path(p).exists() and Path(p).stat().st_size > 0 for p in paths)

### 3.2 The sample table: BAM files, morphs and the binary phenotype

The QC notebook wrote `phenotypes.tsv` with one row per sample, **in the same order as `bam_list.txt`**. That order is critical: ANGSD reads the BAM list and the phenotype file *by line number*, so line $k$ of the phenotype file must describe BAM $k$. We keep the order and **re-derive the BAM paths from the folder we found** (in case you moved the project).

If `phenotypes.tsv` is missing (for example if you only kept the `bams/` folder), we rebuild it from the file names (`<sampleID>.sorted.bam`) and the annotation file `Hypolimnas_misippus_CAMID_morph.txt`.

In [ ]:
pheno_path = PROJECT_DIR / "phenotypes.tsv"
if pheno_path.exists():
    pheno = pd.read_csv(pheno_path, sep="\t")
    print("Read", pheno_path, "->", len(pheno), "samples")
else:
    print("phenotypes.tsv not found: rebuilding the sample table from the BAM names and the morph annotation.")
    morph_file = locate_file("Hypolimnas_misippus_CAMID_morph.txt")
    assert morph_file is not None, "Hypolimnas_misippus_CAMID_morph.txt not found: upload it to the project folder."
    morph = pd.read_csv(morph_file, sep="\t"); morph.columns = ["sampleID", "morph"]
    bams = sorted(BAM_DIR.glob("*.sorted.bam"))
    pheno = pd.DataFrame({"sampleID": [b.name.replace(".sorted.bam", "") for b in bams]}).merge(morph, on="sampleID")

pheno = pheno.copy()
pheno["bam"] = pheno.sampleID.map(lambda s: str(BAM_DIR / f"{s}.sorted.bam"))
pheno["y"] = (pheno.morph == CASE_MORPH).astype(int)           # phenotype used by all three programs: m = 1, i = 0
N = len(pheno)
samples = list(pheno.sampleID)

missing_bam = [b for b in pheno.bam if not Path(b).exists()]
missing_bai = [b for b in pheno.bam if Path(b).exists() and not Path(b + ".bai").exists()]
assert not missing_bam, f"{len(missing_bam)} BAM files are missing, e.g. {missing_bam[:3]}"
assert not missing_bai, f"{len(missing_bai)} BAM files have no .bai index, e.g. {missing_bai[:3]}"
assert pheno.sampleID.is_unique, "duplicated sample IDs"
assert set(pheno.morph) <= {"m", "i"}, "unexpected morph labels"

print(f"\n{N} samples with BAM + index.  Morphs: {pheno.morph.value_counts().to_dict()}")
print(f"Phenotype coding for all programs:  {CASE_MORPH} = 1 (case),  the other morph = 0 (control)")
display(pheno.head())

### 3.3 Integrity checks on the BAM files

Two cheap checks that prevent hours of confusion later:

1. **`samtools quickcheck`** verifies that each BAM has a valid header and end-of-file marker (a truncated copy on Drive is the usual culprit).
2. **The `SM` tag of the read group must equal the sample ID.** The QC notebook set `SM:<sampleID>` in `bwa mem -R`. If the tag disagrees with the file name, a phenotype could be attached to the wrong individual, silently. We compare them for every BAM.

In [ ]:
r = subprocess.run(["samtools", "quickcheck", "-v"] + list(pheno.bam), capture_output=True, text=True)
bad = [l for l in r.stdout.splitlines() if l.strip()]
print("samtools quickcheck:", "all BAM files are OK" if not bad else f"{len(bad)} PROBLEMATIC files, e.g. {bad[:3]}")
assert not bad, "Fix or regenerate the broken BAM files before continuing."

def bam_sample_tag(bam):
    h = subprocess.run(["samtools", "view", "-H", bam], capture_output=True, text=True).stdout
    sm = re.findall(r"^@RG\t.*?SM:([^\t\n]+)", h, flags=re.M)
    return sm[0] if sm else None

pheno["bam_SM"] = [bam_sample_tag(b) for b in pheno.bam]
mismatch = pheno[pheno.bam_SM != pheno.sampleID]
print(f"Read-group SM tag equals the sample ID for {N - len(mismatch)} of {N} BAM files")
assert mismatch.empty, "SM tag and file name disagree for:\n" + mismatch[["sampleID", "bam_SM"]].head().to_string()

### 3.4 Is sequencing depth a confounder?

This repeats, in one cell, the most important check of the QC notebook. If the `i` samples had systematically more or less data than the `m` samples, **missing-data patterns** (hard calls!) and genotype uncertainty would differ between cases and controls, and could create spurious associations. The depth was computed in the QC notebook and stored in `phenotypes.tsv`.

In [ ]:
if "mean_depth" in pheno.columns:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for m_ in MORPH_ORDER:
        v = pheno.loc[pheno.morph == m_, "mean_depth"]
        ax[0].hist(np.log10(v.clip(lower=1e-4)), bins=30, alpha=0.6, color=MORPH_COLORS[m_], label=f"{m_} (n={len(v)})")
    ax[0].set(xlabel="log10 mean depth per sample", ylabel="samples", title="Mean depth by morph"); ax[0].legend()
    sns.boxplot(data=pheno, x="morph", y="mean_depth", order=MORPH_ORDER, hue="morph", palette=MORPH_COLORS,
                showfliers=False, ax=ax[1], legend=False)
    sns.stripplot(data=pheno, x="morph", y="mean_depth", order=MORPH_ORDER, color="k", size=2.5, alpha=0.4, ax=ax[1])
    ax[1].set(xlabel="", ylabel="mean depth (x)", title="Mann-Whitney P = {:.2g}".format(
        stats.mannwhitneyu(pheno.loc[pheno.morph == "m", "mean_depth"], pheno.loc[pheno.morph == "i", "mean_depth"]).pvalue))
    plt.tight_layout(); plt.show()
    print(pheno.groupby("morph").mean_depth.describe()[["count", "min", "50%", "mean", "max"]])
    print(f"\nTotal expected coverage of the dataset: {pheno.mean_depth.sum():.1f}x summed over samples "
          f"(median {pheno.mean_depth.median():.3f}x per sample)")
else:
    print("mean_depth is not in the sample table: skipping.")

### 3.5 Copy the BAM files to the local disk (optional but recommended)

ANGSD reads every BAM many times (depth pilot, calling, three association runs). Reading from Google Drive is slow and occasionally fails with I/O errors. Our BAMs are small subsets, so we copy them (and their indexes) to Colab's local disk once. From now on `pheno.bam` points to the local copies. The copy is skipped for files already present, so re-running is cheap.

In [ ]:
if COPY_BAMS_LOCAL:
    LOCAL_BAMS.mkdir(parents=True, exist_ok=True)
    total = sum(Path(b).stat().st_size + Path(b + ".bai").stat().st_size for b in pheno.bam)
    free = shutil.disk_usage("/content").free
    print(f"BAMs + indexes: {total/1e9:.2f} GB;  free local disk: {free/1e9:.1f} GB")
    if total < 0.5 * free:
        t0 = time.time()
        for i, b in enumerate(pheno.bam):
            for src in (Path(b), Path(b + ".bai")):
                dst = LOCAL_BAMS / src.name
                if not dst.exists() or dst.stat().st_size != src.stat().st_size:
                    shutil.copy2(src, dst)
            if (i + 1) % 50 == 0 or i + 1 == N:
                print(f"  copied {i + 1} / {N} samples ({time.time() - t0:.0f} s)")
        pheno["bam"] = pheno.sampleID.map(lambda s: str(LOCAL_BAMS / f"{s}.sorted.bam"))
        print(f"Copied in {time.time() - t0:.0f} s. Using local BAMs from now on.")
    else:
        print("Not enough local disk: reading the BAMs directly from Drive.")
else:
    print("COPY_BAMS_LOCAL is False: reading the BAMs directly from Drive.")

## 4. Genotype Likelihoods in Practice: a Toy Simulation

Before running ANGSD on the real data, let us convince ourselves with a simulation that **hard genotype calls at low coverage are a bad idea**, and see what a genotype likelihood looks like.

### 4.1 One individual, a few reads

Suppose a site has a minor allele with population frequency $f = 0.3$. We observe 1, 2, 3 or 6 reads from an individual, all with base quality Q30 ($\varepsilon = 0.001$). The cell computes $\mathrm{GL}(g)$, the prior and the posterior, using exactly the formulas of Section 0.2.

In [ ]:
def genotype_likelihoods(n_minor, n_major, err=1e-3):
    # GL for g = 0, 1, 2 copies of the minor allele, given n_minor reads showing the minor allele and n_major the major allele.
    g = np.array([0, 1, 2])
    p_minor = g / 2 * (1 - err) + (1 - g / 2) * err          # probability that one read shows the minor allele
    return p_minor ** n_minor * (1 - p_minor) ** n_major

def genotype_posterior(gl, f):
    prior = np.array([(1 - f) ** 2, 2 * f * (1 - f), f ** 2])   # Hardy-Weinberg
    post = gl * prior
    return prior, post / post.sum()

f = 0.3
rows = []
for n_min, n_maj in [(0, 0), (1, 0), (0, 1), (1, 1), (2, 1), (3, 3), (6, 0), (3, 0)]:
    gl = genotype_likelihoods(n_min, n_maj)
    prior, post = genotype_posterior(gl, f)
    rows.append(dict(reads_minor=n_min, reads_major=n_maj,
                     GL_0=gl[0], GL_1=gl[1], GL_2=gl[2],
                     post_0=post[0], post_1=post[1], post_2=post[2],
                     dosage=post[1] + 2 * post[2], best_call=int(post.argmax()), confidence=post.max()))
toy = pd.DataFrame(rows)
display(toy.style.format({c: "{:.3g}" for c in ["GL_0", "GL_1", "GL_2"]} | {c: "{:.3f}" for c in ["post_0", "post_1", "post_2", "dosage", "confidence"]}))
print("Read the table row by row:")
print(" * (0,0): no reads -> the posterior equals the prior; the genotype is completely unknown.")
print(" * (1,0): ONE read of the minor allele. Even with a perfect read, the posterior of the best genotype is far from 0.9.")
print(" * (6,0): six reads, all minor -> now a confident homozygote.")
print(" * (3,3): both alleles seen -> a confident heterozygote (errors cannot produce 3 minor reads at Q30).")

### 4.2 How much of the genome can be hard-called at a given coverage?

We now simulate many individuals with true genotypes drawn from Hardy-Weinberg, sample the number of reads from a Poisson distribution with mean equal to the coverage, assign each read to one of the two chromosomes at random, add errors, and compute the posterior. We call the genotype only if the posterior is $\ge 0.9$ (the cut-off we will use for PLINK). The plot shows

* the **fraction of genotypes that get called** (the rest become missing), and
* the **fraction of called genotypes that are correct**, separately for homozygotes and heterozygotes.

In [ ]:
rng = np.random.default_rng(1)

def simulate_calls(mean_depth, f=0.3, n=20000, err=1e-3, cutoff=0.9):
    g = rng.binomial(2, f, size=n)                               # true number of minor alleles
    depth = rng.poisson(mean_depth, size=n)
    # reads: each read comes from one of the two chromosomes (minor with prob g/2), and is wrong with prob err
    n_minor_chr = rng.binomial(depth, g / 2)
    n_major_chr = depth - n_minor_chr
    flip_minor = rng.binomial(n_minor_chr, err)                  # minor reads that look major
    flip_major = rng.binomial(n_major_chr, err)                  # major reads that look minor
    n_min = n_minor_chr - flip_minor + flip_major
    n_maj = depth - n_min
    G = np.array([0, 1, 2])
    p_minor = G / 2 * (1 - err) + (1 - G / 2) * err
    gl = (p_minor[None, :] ** n_min[:, None]) * ((1 - p_minor[None, :]) ** n_maj[:, None])
    prior = np.array([(1 - f) ** 2, 2 * f * (1 - f), f ** 2])
    post = gl * prior; post /= post.sum(axis=1, keepdims=True)
    call = post.argmax(axis=1); conf = post.max(axis=1)
    called = conf >= cutoff
    out = dict(depth=mean_depth, frac_called=called.mean())
    for name, gg in [("hom", (g == 0) | (g == 2)), ("het", g == 1)]:
        sel = gg & called
        out[f"frac_{name}_called"] = (called[gg]).mean()
        out[f"acc_{name}"] = (call[sel] == g[sel]).mean() if sel.sum() else np.nan
    return out

depths = [0.05, 0.1, 0.2, 0.4, 0.8, 1.5, 3, 6, 10, 20]
sim = pd.DataFrame([simulate_calls(d) for d in depths])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(sim.depth, sim.frac_called, "k-o", label="all genotypes")
ax[0].plot(sim.depth, sim.frac_hom_called, "-o", c="#4c72b0", label="true homozygotes")
ax[0].plot(sim.depth, sim.frac_het_called, "-o", c="#c44e52", label="true heterozygotes")
ax[0].axvline(0.81, c="grey", ls="--"); ax[0].text(0.85, 0.5, "paper: ~0.81x", rotation=90, color="grey")
ax[0].set(xscale="log", xlabel="mean depth per individual (x)", ylabel="fraction of genotypes with posterior >= 0.9",
          title="How much survives hard-calling?"); ax[0].legend()
ax[1].plot(sim.depth, sim.acc_hom, "-o", c="#4c72b0", label="homozygotes")
ax[1].plot(sim.depth, sim.acc_het, "-o", c="#c44e52", label="heterozygotes")
ax[1].set(xscale="log", xlabel="mean depth per individual (x)", ylabel="fraction of called genotypes that are correct",
          title="Accuracy of the genotypes that ARE called", ylim=(0.85, 1.01)); ax[1].legend()
plt.tight_layout(); plt.show()
display(sim.round(3))

**What the simulation teaches.**

* Below ~1x the **large majority of genotypes are missing** after hard-calling (see the left panel and the table: at 0.8x only a few per cent of the genotypes reach a posterior of 0.9).
* The genotypes that *are* called are not all correct, and the errors are **not symmetric**. A heterozygote is called with confidence when reads from *both* chromosomes are seen. But a true heterozygote whose few reads all happen to come from the same chromosome looks like a confident homozygote, so the accuracy for heterozygotes (right panel) is the lowest. Hard-called low-coverage data therefore carry an **excess of homozygotes**, not because the population is inbred but because of the calling procedure.
* The genotypes that survive are the ones from **well-covered sites and individuals**, so the surviving data are not a random sample.
* Methods that keep the uncertainty (genotype likelihoods, posteriors, dosages) do not have this problem: every individual contributes at every site, in proportion to the information it carries.

This is the reason why we first call variants with ANGSD, and then compare three ways of passing the result to association software.

## 5. Chromosomes: Which Parts of the Genome Do We Analyse?

We need (a) the list of chromosomes and their lengths, (b) the **name of chromosome 26** (the M locus), and (c) the names of the **Z and W** chromosomes so that we can exclude them (females are ZW).

The reference FASTA headers contain descriptions like `... chromosome: 26`. We read the contig lengths from the BAM header (always available) and the labels from the reference FASTA in `reference_HypMis_v2/` (if present), exactly as in Section 9.5 of the QC notebook.

In [ ]:
# ---- contig names and lengths from the header of the first BAM
hdr = subprocess.run(["samtools", "view", "-H", pheno.bam.iloc[0]], capture_output=True, text=True).stdout
sq = [l.split("\t") for l in hdr.splitlines() if l.startswith("@SQ")]
fai = pd.DataFrame({"contig": [x[1][3:] for x in sq], "length": [int(x[2][3:]) for x in sq]})

# ---- chromosome labels from the FASTA headers (optional)
ref_candidates = sorted((PROJECT_DIR / "reference_HypMis_v2").glob("*.fa")) if (PROJECT_DIR / "reference_HypMis_v2").exists() else []
if ref_candidates:
    REF_FASTA = ref_candidates[0]
    heads = subprocess.run(["grep", ">", str(REF_FASTA)], capture_output=True, text=True).stdout.splitlines()
    desc = {h[1:].split(" ", 1)[0]: (h[1:].split(" ", 1) + [""])[1] for h in heads}
    fai["description"] = fai.contig.map(desc)
    fai["label"] = fai.description.str.extract(r"(?:chromosome|scaffold)[:\s_]+([A-Za-z0-9]+)", expand=False)
    print("Reference FASTA:", REF_FASTA)
else:
    REF_FASTA = None
    fai["description"] = ""; fai["label"] = np.nan
    print("Reference FASTA not found: chromosome labels are unavailable. Set M_CHROM / Z_CHROM manually below.")

def contig_by_label(label):
    hit = fai.loc[fai.label == str(label), "contig"]
    return hit.iloc[0] if len(hit) else None

M_CHROM = contig_by_label("26")        # set manually if None or wrong, e.g. M_CHROM = "OZ123456.1"
Z_CHROM = contig_by_label("Z")
W_CHROM = contig_by_label("W")

big = fai[fai.length > 5e6]
autosome_mask = big.label.fillna("").str.fullmatch(r"\d+") & ~big.contig.isin([c for c in [Z_CHROM, W_CHROM] if c])
AUTOSOMES = list(big.loc[autosome_mask, "contig"])
if not AUTOSOMES:                       # fallback: no labels -> every sequence > 5 Mb except the Z
    AUTOSOMES = [c for c in big.contig if c != Z_CHROM]
    print("No numeric chromosome labels found: using all sequences > 5 Mb except the Z chromosome.")
ANALYSED = AUTOSOMES + ([Z_CHROM] if (INCLUDE_Z and Z_CHROM) else [])
ANALYSED = [c for c in fai.contig if c in set(ANALYSED)]                       # keep BAM-header order

fai = fai.set_index("contig", drop=False)
CONTIG_HDR = D["vcf"] / "contigs.hdr"          # '##contig' header lines, added to VCFs that lack them
CONTIG_HDR.write_text("".join(f"##contig=<ID={c},length={l}>\n" for c, l in zip(fai.contig, fai.length)))
OFFSET = {}; cum = 0
for c in fai.contig:
    OFFSET[c] = cum; cum += int(fai.loc[c, "length"])      # cumulative coordinates, used for Manhattan plots and SNPTEST

print(f"\nSequences in the BAM header : {len(fai)}")
print(f"M chromosome (26)           : {M_CHROM}")
print(f"Z chromosome / W chromosome : {Z_CHROM} / {W_CHROM}")
print(f"Chromosomes analysed        : {len(ANALYSED)} ({fai.loc[ANALYSED, 'length'].sum()/1e6:.1f} Mb of {fai.length.sum()/1e6:.1f} Mb)")
assert M_CHROM is not None, "Chromosome 26 not identified: set M_CHROM manually above (it is the one carrying the M locus)."
assert M_CHROM in ANALYSED, "M_CHROM is not in the analysed chromosomes: check the labels."
display(fai.loc[ANALYSED, ["contig", "length", "label"]].reset_index(drop=True).T)

### 5.1 Input files for ANGSD

ANGSD wants plain text files:

| File | Format | Notes |
|---|---|---|
| `bam_list.txt` | one BAM path per line | **defines the individual order** for everything else |
| `pheno.ybin` | one value per line: `1` = case, `0` = control, `-999` = missing | same order as the BAM list; used by `-yBin` |
| `regions.txt` | one contig name per line | restricts the analysis with `-rf` (autosomes only) |
| `covariates.txt` | one row per individual, numeric columns | only if `N_PCS_COV > 0`; used by `-cov` |

We write them once, with the sample order stored in `samples_in_order.txt` so that we can rename the samples in the ANGSD output later.

In [ ]:
BAM_LIST = D["angsd"] / "bam_list.txt"
YBIN = D["angsd"] / "pheno.ybin"
REGIONS = D["angsd"] / "regions.txt"
SAMPLE_ORDER = D["angsd"] / "samples_in_order.txt"

BAM_LIST.write_text("\n".join(pheno.bam) + "\n")
YBIN.write_text("\n".join(map(str, pheno.y)) + "\n")
REGIONS.write_text("\n".join(ANALYSED) + "\n")
SAMPLE_ORDER.write_text("\n".join(samples) + "\n")

print(f"{BAM_LIST.name}: {N} lines;  {YBIN.name}: {pheno.y.sum()} cases (1) and {(pheno.y == 0).sum()} controls (0);  "
      f"{REGIONS.name}: {len(ANALYSED)} chromosomes")
print("\nFirst three lines side by side (BAM  |  phenotype):")
for b, y in zip(pheno.bam[:3], pheno.y[:3]):
    print(f"  {b}  |  {y}")

## 6. A Depth Pilot with ANGSD: Choosing the Site Filters

Two site-level filters depend on the data and should not be copied blindly from another project:

* **`-minInd`**: the minimum number of individuals that must have **at least one read** at a site. If it is too low, the allele frequency and the SNP test rest on a handful of individuals. If it is too high, we discard most of the genome at low coverage.
* **`-setMaxDepth`**: the maximum **total** depth (summed over all individuals) at a site. Sites with an abnormally high depth are typically **collapsed repeats or paralogues**, where reads from several genomic copies pile up on one reference position and create fake SNPs. Transposable elements, which are the very cause of the M-locus polymorphism, are exactly this kind of sequence.

We measure both on one chromosome, the M chromosome (26), with `-doDepth 1`. ANGSD then writes

* `*.depthSample`: for each individual, the number of sites at depth 0, 1, 2, ..., `maxDepth`;
* `*.depthGlobal`: the same for the sum over all individuals.

| Flag | Meaning |
|---|---|
| `-b` | file with the BAM paths |
| `-r chr:` | restrict to one chromosome (the colon means "the whole of it") |
| `-doCounts 1` | count the bases at every site (needed for depth) |
| `-doDepth 1` | write the depth distributions |
| `-maxDepth 1000` | depth values above this are pooled in the last bin |
| `-minMapQ`, `-minQ` | discard reads / bases of low mapping / base quality |
| `-uniqueOnly 1` | only reads with a unique best hit |
| `-remove_bads 1` | remove reads flagged as duplicate, failing QC, etc. |
| `-only_proper_pairs 0` | our reads are **single-end**, so there are no proper pairs to require |

In [ ]:
PILOT = D["angsd"] / "pilot_depth"
MAXD_PILOT = 1000
READ_FILTERS = (f"-minMapQ {MIN_MAPQ} -minQ {MIN_BASEQ} -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 0")

if not done(f"{PILOT}.depthSample", f"{PILOT}.depthGlobal"):
    run(f"angsd -b {BAM_LIST} -r {M_CHROM}: -doCounts 1 -doDepth 1 -maxDepth {MAXD_PILOT} {READ_FILTERS} "
        f"-nThreads {THREADS} -out {PILOT}", log=D["angsd"] / "pilot_depth.log")
else:
    print("Depth pilot already done: reading", PILOT)
print([p.name for p in sorted(D["angsd"].glob("pilot_depth*"))])

### 6.1 Reading the pilot output

From `depthSample` we compute for each individual

* the **fraction of sites at which it has data** (at least one read), relative to the number of sites at which *anyone* has data. Summed over individuals this is the **expected number of individuals with data at a covered site**, the quantity that `-minInd` acts on;
* its mean depth on this chromosome, which we compare with the value from the QC notebook as a sanity check of the whole pipeline.

From `depthGlobal` we get the distribution of the total depth and set the upper cut-off to **mean + 3 standard deviations**, a standard rule of thumb for removing the upper tail.

> **Rule used for `-minInd`.** We require data in *half of the individuals that are expected to be covered*: `MIN_IND = 0.5 x (expected number of covered individuals)`, bounded between 3 and N/2. At 0.8x average depth about 55% of individuals have at least one read at a site, so this gives roughly 0.27 N. You can overrule the rule by setting `MIN_IND` (and `MAX_DEPTH`) in Section 3.1.

In [ ]:
ds = np.atleast_2d(np.loadtxt(f"{PILOT}.depthSample"))
dg = np.loadtxt(f"{PILOT}.depthGlobal").ravel()
assert ds.shape[0] == N, f"depthSample has {ds.shape[0]} rows but there are {N} samples"
dvals = np.arange(ds.shape[1])

sites_any = dg[1:].sum()                                   # sites with data in at least one individual
cov_frac = ds[:, 1:].sum(axis=1) / sites_any               # per individual: fraction of those sites with data
chrom_len = int(fai.loc[M_CHROM, "length"])
pilot_depth = (ds * dvals).sum(axis=1) / chrom_len         # mean depth over the whole chromosome

tot = np.arange(len(dg))[1:]; w = dg[1:]
mean_tot = (tot * w).sum() / w.sum()
sd_tot = np.sqrt(((tot - mean_tot) ** 2 * w).sum() / w.sum())
max_depth_auto = int(min(np.ceil(mean_tot + 3 * sd_tot), MAXD_PILOT))
exp_ind = cov_frac.sum()
min_ind_auto = int(np.clip(round(0.5 * exp_ind), 3, N // 2))

MIN_IND = MIN_IND if MIN_IND is not None else min_ind_auto
MAX_DEPTH = MAX_DEPTH if MAX_DEPTH is not None else max_depth_auto

print(f"Sites on {M_CHROM} with data in at least one individual : {int(sites_any):,} of {chrom_len:,} bp "
      f"({100*sites_any/chrom_len:.1f} %)")
print(f"Expected number of individuals with data at such a site  : {exp_ind:.1f} of {N}")
print(f"Total depth at covered sites: mean {mean_tot:.1f}, sd {sd_tot:.1f}  ->  mean + 3 sd = {mean_tot + 3*sd_tot:.0f}")
print(f"Fraction of sites in the top (pooled) depth bin         : {dg[-1] / max(dg[1:].sum(), 1):.4f}")
print(f"\n==> -minInd = {MIN_IND}    (rule: {min_ind_auto})")
print(f"==> -setMaxDepth = {MAX_DEPTH}    (rule: {max_depth_auto})")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for m_ in MORPH_ORDER:
    sel = (pheno.morph == m_).values
    ax[0].hist(cov_frac[sel], bins=30, alpha=0.6, color=MORPH_COLORS[m_], label=m_)
ax[0].set(xlabel="fraction of covered sites at which the individual has data", ylabel="individuals",
          title="Per-individual data availability"); ax[0].legend()
shown = min(len(dg), int(mean_tot + 6 * sd_tot) + 10)
ax[1].bar(np.arange(1, shown), dg[1:shown], width=1.0, color="grey")
ax[1].axvline(MAX_DEPTH, c="r", ls="--", label=f"-setMaxDepth {MAX_DEPTH}")
ax[1].set(yscale="log", xlabel="total depth at a site (all individuals)", ylabel="number of sites",
          title="Global depth distribution"); ax[1].legend()
if "mean_depth" in pheno.columns:
    ax[2].scatter(pheno.mean_depth, pilot_depth, s=10, c=pheno.morph.map(MORPH_COLORS))
    lim = [min(pheno.mean_depth.min(), pilot_depth.min()) * 0.8, max(pheno.mean_depth.max(), pilot_depth.max()) * 1.2]
    ax[2].plot(lim, lim, "k--", lw=1)
    ax[2].set(xscale="log", yscale="log", xlabel="mean depth, QC notebook (genome-wide)",
              ylabel="mean depth, ANGSD pilot (this chromosome)", title="Consistency check\n(ANGSD filters are stricter, so slightly lower)")
plt.tight_layout(); plt.show()

## 7. Probabilistic Variant Calling with ANGSD

### 7.1 The command

We now run ANGSD on all BAM files and all analysed chromosomes. One run produces **everything** the later steps need:

| Flag | Meaning |
|---|---|
| `-b bam_list.txt`, `-rf regions.txt` | the individuals and the chromosomes to analyse |
| `-GL 2` | genotype likelihood model 2 (GATK-style) |
| `-doMajorMinor 1` | infer the major and minor allele from the genotype likelihoods |
| `-doMaf 1` | estimate the minor-allele frequency by EM |
| `-SNP_pval 1e-6` | call a site a SNP only if the LRT against "frequency = 0" gives P below this |
| `-minMaf 0.05` | keep SNPs with minor-allele frequency above 5% (common variants only: GWAS power) |
| `-minInd`, `-setMaxDepth`, `-doCounts 1` | the site filters from the pilot |
| `-minMapQ 30 -minQ 20 -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 0` | read filters |
| `-doPost 1` | genotype **posteriors**, with the estimated allele frequency as prior (needed for association, BCF and SNPTEST) |
| `-doGlf 2` | write the genotype likelihoods in **BEAGLE** format (input of pcangsd) |
| `-doBcf 1` | write a **BCF** with `GL` and `GP` for every sample (ANGSD's older `-doVcf` is deprecated) |
| `-nThreads` | threads |

> **Why `-minMaf 0.05`?** In a GWAS a variant with 3 copies of the rare allele in 332 individuals has no statistical power and large sampling variance. We follow the ANGSD lab and the common practice of using MAF > 5%. The morph `i` is 17% of the sample, and a variant tightly linked to the recessive allele `m` will have a frequency well above 5%.
>
> **What we do *not* use.** Base-alignment quality recalibration (`-baq`, `-C 50`) needs the reference FASTA (`-ref`); it matters for indels. We use the SAMtools/GATK GL without it to keep the notebook light.

In [ ]:
CALL_PREFIX = D["angsd"] / "HypMis_calls"
ANGSD_COMMON = (f"-b {BAM_LIST} -rf {REGIONS} -GL {GL_MODEL} -doMajorMinor 1 -doMaf 1 -doCounts 1 "
                f"{READ_FILTERS} -minInd {MIN_IND} -setMaxDepth {MAX_DEPTH} "
                f"-SNP_pval {SNP_PVAL} -minMaf {MIN_MAF} -nThreads {THREADS}")

if not done(f"{CALL_PREFIX}.beagle.gz", f"{CALL_PREFIX}.mafs.gz") or not glob.glob(f"{CALL_PREFIX}.bcf") + glob.glob(f"{CALL_PREFIX}.vcf*"):
    run(f"angsd {ANGSD_COMMON} -doPost 1 -doGlf 2 -doBcf 1 -out {CALL_PREFIX}", log=D["angsd"] / "angsd_calls.log", show=8)
else:
    print("ANGSD calls already exist: skipping the run.")

CALLS_RAW = (glob.glob(f"{CALL_PREFIX}.bcf") + glob.glob(f"{CALL_PREFIX}.vcf.gz") + glob.glob(f"{CALL_PREFIX}.vcf"))
assert CALLS_RAW and Path(f"{CALL_PREFIX}.beagle.gz").exists(), (
    "ANGSD did not produce the expected files. See association_HypMis/angsd/angsd_calls.log. "
    "Typical causes: no site passed the filters (lower MIN_IND or MIN_MAF), or a BAM could not be read.")
CALLS_RAW = CALLS_RAW[0]
print("\nOutput files:")
for p in sorted(D["angsd"].glob("HypMis_calls*")):
    print(f"  {p.name:35s} {p.stat().st_size/1e6:10.2f} MB")

### 7.2 The three ANGSD output files

**`.mafs.gz`**: one row per SNP: chromosome, position, major allele, minor allele, reference ("N" because we gave no reference), the EM estimate of the minor-allele frequency (`knownEM`) and the number of individuals with data (`nInd`). With `-SNP_pval` there is also the p-value of the SNP test (`pK-EM`).

**`.beagle.gz`**: a text matrix with a header (`marker allele1 allele2 Ind0 Ind0 Ind0 Ind1 ...`) and, for each SNP, **three numbers per individual**: the (normalised) genotype likelihoods of `AA`, `AB`, `BB`, where `A` is the major and `B` the minor allele. An individual with no reads gets `0.333 0.333 0.333`. The alleles are coded 0, 1, 2, 3 = A, C, G, T. This is the input of pcangsd.

**`.bcf`**: the same information in VCF form, with the posteriors (`GP`) and log-likelihoods (`GL`). Let us look at the first two files.

In [ ]:
mafs = pd.read_csv(f"{CALL_PREFIX}.mafs.gz", sep="\t")
mafs = mafs.loc[:, ~mafs.columns.str.startswith("Unnamed")]
mafs = mafs.rename(columns={"chromo": "contig", "position": "pos", "knownEM": "maf"})
print(f"{len(mafs):,} SNPs called by ANGSD on {mafs.contig.nunique()} chromosomes")
assert len(mafs) > 0, "No SNP passed the filters: lower MIN_IND / MIN_MAF / raise SNP_PVAL (Section 3.1) and re-run."
print("\n--- first rows of the .mafs file ---")
display(mafs.head())

print("--- first rows of the .beagle file (3 numbers per individual; first 2 individuals shown) ---")
with gzip.open(f"{CALL_PREFIX}.beagle.gz", "rt") as fh:
    rows = [fh.readline().rstrip("\n").split("\t") for _ in range(4)]
display(pd.DataFrame(rows[1:], columns=rows[0])[rows[0][:9]])

In [ ]:
# What do the SNPs look like? number per chromosome, MAF spectrum, number of individuals with data
per_chr = mafs.groupby("contig").size().reindex(ANALYSED).fillna(0)
lab = fai.loc[ANALYSED, "label"].fillna("?")
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].bar(range(len(per_chr)), per_chr.values, color="grey")
ax[0].set_xticks(range(len(per_chr))); ax[0].set_xticklabels(lab, rotation=90, fontsize=7)
ax[0].set(xlabel="chromosome", ylabel="SNPs", title="SNPs per chromosome")
ax[1].hist(mafs.maf, bins=40, color="grey")
ax[1].set(xlabel="minor allele frequency (ANGSD EM estimate)", ylabel="SNPs", title="Allele-frequency spectrum")
ax[2].hist(mafs.nInd, bins=40, color="grey"); ax[2].axvline(MIN_IND, c="r", ls="--", label=f"-minInd {MIN_IND}")
ax[2].set(xlabel="individuals with data at the SNP", ylabel="SNPs", title="Data per SNP"); ax[2].legend()
plt.tight_layout(); plt.show()

dens = per_chr / fai.loc[ANALYSED, "length"] * 1e6
print(f"Median SNP density: {dens.median():.1f} SNPs per Mb;  total {len(mafs):,} SNPs over {fai.loc[ANALYSED, 'length'].sum()/1e6:.0f} Mb")
n_m = int((mafs.contig == M_CHROM).sum()); n_win = int(((mafs.contig == M_CHROM) & mafs.pos.between(PEAK_START, PEAK_END)).sum())
print(f"On the M chromosome: {n_m:,} SNPs, of which {n_win} inside the associated window {PEAK_START:,}-{PEAK_END:,}")

### 7.3 From BCF to an indexed VCF with the right sample names

ANGSD writes a **binary BCF**; we convert it with BCFtools to a **bgzipped VCF** (`.vcf.gz`) plus its index (`.tbi`), the form that every downstream program understands. Two clean-up steps:

1. **Sample names.** ANGSD labels the columns with the BAM file names (or a generic name). We rename them to the CAM sample IDs in the exact order of the BAM list (`bcftools reheader -s`). Because the order of the BAM list is the order of the columns, this is safe, and it is why we saved `samples_in_order.txt`.
2. **Contig lines.** A well-formed VCF header lists the contigs (`##contig=<ID=...,length=...>`). If ANGSD did not write them we add them from the BAM header; BCFtools and tabix need them for region queries.

In [ ]:
CALLS_VCF = D["vcf"] / "HypMis_ANGSD_calls.vcf.gz"
if not done(CALLS_VCF, str(CALLS_VCF) + ".tbi"):
    n_cols = int(run(f"bcftools query -l {CALLS_RAW} | wc -l", quiet=True).stdout.strip())
    assert n_cols == N, f"ANGSD output has {n_cols} samples but the BAM list has {N}."
    renamed = WORK / "angsd_renamed.bcf"
    run(f"bcftools reheader -s {SAMPLE_ORDER} -o {renamed} {CALLS_RAW}", quiet=True)
    has_contigs = int(run(f"bcftools view -h {renamed} | grep -c '^##contig' || true", quiet=True).stdout.strip() or 0)
    if has_contigs == 0:
        run(f"bcftools annotate -h {CONTIG_HDR} -Oz -o {CALLS_VCF} {renamed}", quiet=True)
    else:
        run(f"bcftools view -Oz -o {CALLS_VCF} {renamed}", quiet=True)
    run(f"bcftools index -t -f {CALLS_VCF}", quiet=True)
n_calls = int(run(f"bcftools view -H {CALLS_VCF} | wc -l", quiet=True).stdout.strip())
print(f"{CALLS_VCF.name}: {n_calls:,} records (the 'universe' of variants for the rest of the notebook)")
if n_calls != len(mafs):
    print(f"NOTE: the VCF has {n_calls:,} records but the .mafs file has {len(mafs):,} SNPs. The VCF is used as the universe from here on.")

print("\n--- header lines describing INFO and FORMAT fields ---")
print(run(f"bcftools view -h {CALLS_VCF} | grep -E '^##(INFO|FORMAT)'", quiet=True).stdout)
print("--- first two records (first 12 columns; the sample columns carry GP and GL) ---")
print(run(f"bcftools view -H {CALLS_VCF} | head -n 2 | cut -f 1-12", quiet=True).stdout)

### 7.4 Reading a record: what do `GP` and `GL` contain?

In the VCF `GP` is the **genotype posterior probability** for `0/0`, `0/1`, `1/1` and `GL` is the log10 genotype likelihood for the same three genotypes. `REF` is ANGSD's **major** allele and `ALT` the **minor** allele (we did not give a reference sequence), so `1/1` means "homozygous for the minor allele" and the dosage $E[G\mid D] = GP_{0/1} + 2\,GP_{1/1}$ counts minor alleles.

Because all later conversions (hard calls for PLINK, `.gen` for SNPTEST) are built from `GP`, we write a small reader that streams the VCF with `bcftools query` and yields, for every variant, an array of shape (individuals, 3). We first check that `REF`/`ALT` agree with the major/minor alleles in the `.mafs` file, and then measure **how confident the genotypes are**: for every individual, the fraction of SNPs where the best genotype has posterior $\ge$ `POST_CUTOFF`. This is the real-data version of Section 4.2.

In [ ]:
def parse_gp_fields(rest):
    # rest looks like ',a,b,c,a,b,c,...' (one triplet per individual); returns an array of shape (n_individuals, 3)
    s = rest[1:]
    try:
        arr = np.array(s.split(","), dtype=np.float32)
    except ValueError:                                    # missing values appear as '.'
        arr = np.array([float(x) if x not in (".", "") else np.nan for x in s.split(",")], dtype=np.float32)
    return arr.reshape(-1, 3)

def iter_gp(vcf):
    # Stream (chrom, pos, ref, alt, GP) for every record of a VCF/BCF.
    p = subprocess.Popen(["bcftools", "query", "-f", "%CHROM %POS %REF %ALT[,%GP]\n", str(vcf)],
                         stdout=subprocess.PIPE, text=True, bufsize=1 << 20)
    for line in p.stdout:
        chrom, pos, ref, alt, rest = line.rstrip("\n").split(" ", 4)
        yield chrom, int(pos), ref, alt, parse_gp_fields(rest)
    p.stdout.close()
    if p.wait() != 0:
        raise RuntimeError("bcftools query failed")

conf_per_ind = np.zeros(N); n_sites = 0; conf_per_site = []; ok_alleles = 0
maj_min = dict(zip(zip(mafs.contig, mafs.pos), zip(mafs.major, mafs.minor)))
t0 = time.time()
for chrom, pos, ref, alt, gp in iter_gp(CALLS_VCF):
    assert gp.shape[0] == N, f"{chrom}:{pos} has {gp.shape[0]} GP triplets instead of {N}"
    c = gp.max(axis=1) >= POST_CUTOFF
    conf_per_ind += c; conf_per_site.append(c.mean()); n_sites += 1
    mm = maj_min.get((chrom, pos))
    if mm is not None and (ref, alt) == (str(mm[0]), str(mm[1])):
        ok_alleles += 1
conf_per_ind /= n_sites
conf_per_site = np.array(conf_per_site)
print(f"Read {n_sites:,} records in {time.time() - t0:.0f} s")
print(f"REF/ALT equal the major/minor alleles of the .mafs file at {100*ok_alleles/n_sites:.1f} % of the SNPs "
      f"(should be ~100 %; if not, GP is not in major/minor order and the conversions below would be wrong)")
print(f"Median fraction of genotypes with posterior >= {POST_CUTOFF}: {np.median(conf_per_ind):.3f} per individual")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
if "mean_depth" in pheno.columns:
    ax[0].scatter(pheno.mean_depth, conf_per_ind, s=10, c=pheno.morph.map(MORPH_COLORS))
    ax[0].set(xscale="log", xlabel="mean depth of the individual", ylabel=f"fraction of SNPs with posterior >= {POST_CUTOFF}",
              title="Confident genotypes grow with depth")
ax[1].hist(conf_per_ind, bins=30, color="grey")
ax[1].set(xlabel=f"fraction of SNPs with a confident genotype (per individual)", ylabel="individuals", title="Per individual")
ax[2].hist(conf_per_site, bins=40, color="grey")
ax[2].set(xlabel=f"fraction of individuals with a confident genotype (per SNP)", ylabel="SNPs", title="Per SNP")
plt.tight_layout(); plt.show()
print("Everything to the left of 1.0 in these plots is information that PLINK's hard calls will treat as MISSING.")

## 8. Population Structure: PCA with pcangsd

### 8.1 Why PCA before association?

The PCA tells us whether the butterflies form genetically distinct groups. The sample comes from across Africa, so geography is the first thing to expect. It also catches technical effects: at low coverage the first PCs are sometimes **driven by sequencing depth** rather than by ancestry.

`pcangsd` takes the BEAGLE file and

1. estimates the allele frequencies and the individual genotype *posteriors* iteratively (using the PCs themselves as a refined prior),
2. computes the **covariance matrix** between individuals from the expected genotypes with the proper correction for the uncertainty on the diagonal (Meisner & Albrechtsen 2018),
3. writes it to `prefix.cov`.

We then decompose the matrix ourselves with an eigendecomposition. The ANGSD lab does this in R (`e <- eigen(C)`); we do the same with NumPy so the notebook needs no R-Python bridge:

```r
C <- as.matrix(read.table("pca.cov")); e <- eigen(C); plot(e$vectors[, 1:2])
```

| Flag | Meaning |
|---|---|
| `-b` | BEAGLE genotype-likelihood file |
| `-o` | output prefix |
| `-t` | threads |

(The lab also ran NGSadmix; we deliberately skip admixture in this course.)

In [ ]:
PCA_PREFIX = D["pcangsd"] / "pca"
if not done(f"{PCA_PREFIX}.cov"):
    run(f"pcangsd -b {CALL_PREFIX}.beagle.gz -o {PCA_PREFIX} -t {THREADS}", log=D["pcangsd"] / "pcangsd.log", show=10)
else:
    print("pcangsd output already exists: skipping the run.")
print(run(f"ls -lh {PCA_PREFIX}*", quiet=True).stdout)

In [ ]:
C = np.loadtxt(f"{PCA_PREFIX}.cov")
assert C.shape == (N, N), f"covariance matrix is {C.shape} but there are {N} individuals"
evals, evecs = np.linalg.eigh(C)
order = np.argsort(evals)[::-1]
evals, evecs = evals[order], evecs[:, order]
var_expl = np.clip(evals, 0, None) / np.clip(evals, 0, None).sum()

NPC = 10
pcs = pd.DataFrame(evecs[:, :NPC], columns=[f"PC{i+1}" for i in range(NPC)])
pcs.insert(0, "sampleID", samples)
pcs = pcs.merge(pheno[[c for c in ["sampleID", "morph", "y", "mean_depth"] if c in pheno.columns]], on="sampleID")
pcs.to_csv(D["pcangsd"] / "pcs.tsv", sep="\t", index=False)

fig, ax = plt.subplots(1, 4, figsize=(19, 4))
ax[0].bar(range(1, NPC + 1), 100 * var_expl[:NPC], color="grey")
ax[0].set(xlabel="principal component", ylabel="% of variance", title="Scree plot")
for a, (i, j) in zip(ax[1:3], [(1, 2), (3, 4)]):
    for m_ in MORPH_ORDER:
        s = pcs.morph == m_
        a.scatter(pcs.loc[s, f"PC{i}"], pcs.loc[s, f"PC{j}"], s=14, alpha=0.75, color=MORPH_COLORS[m_], label=f"{m_} (n={s.sum()})")
    a.set(xlabel=f"PC{i} ({100*var_expl[i-1]:.1f} %)", ylabel=f"PC{j} ({100*var_expl[j-1]:.1f} %)", title=f"PC{i} vs PC{j}")
    a.legend()
if "mean_depth" in pcs.columns:
    ax[3].scatter(pcs.mean_depth, pcs.PC1, s=10, c=pcs.morph.map(MORPH_COLORS))
    ax[3].set(xscale="log", xlabel="mean depth", ylabel="PC1", title="Is PC1 just depth?")
plt.tight_layout(); plt.show()

### 8.2 Is the structure related to the phenotype or to depth?

Two questions decide whether PCs must be used as covariates:

* **Do the PCs differ between `m` and `i`?** (Mann-Whitney test per PC.) If a PC separates the morphs, then ancestry and phenotype are correlated, and unadjusted tests may be inflated.
* **Does a PC track sequencing depth?** (Spearman correlation.) If so, the PC is partly technical.

The paper reported **no population structure along chromosome 26**, which is why the morph can be mapped with a simple test. The table below shows what is true for *our* subset.

In [ ]:
rows = []
for k in range(1, NPC + 1):
    a = pcs.loc[pcs.morph == "m", f"PC{k}"]; b = pcs.loc[pcs.morph == "i", f"PC{k}"]
    row = dict(PC=f"PC{k}", pct_variance=100 * var_expl[k - 1], P_morph_MWU=stats.mannwhitneyu(a, b).pvalue)
    if "mean_depth" in pcs.columns:
        rho, p = stats.spearmanr(pcs[f"PC{k}"], pcs.mean_depth); row.update(rho_depth=rho, P_depth=p)
    rows.append(row)
pc_tab = pd.DataFrame(rows)
fmt = {"pct_variance": "{:.2f}", "P_morph_MWU": "{:.3g}", "rho_depth": "{:.2f}", "P_depth": "{:.3g}"}
display(pc_tab.style.format({k: v for k, v in fmt.items() if k in pc_tab.columns}))
flag = pc_tab[pc_tab.P_morph_MWU < 0.05 / NPC]
print("PCs separating the morphs after Bonferroni correction over 10 PCs:", list(flag.PC) if len(flag) else "none")
print("\nDecision rule used in the course: if the first PCs separate the morphs (or the QQ plots of Sections 9-11 show lambda > 1.1),")
print("set N_PCS_COV = 2 or 3 in Section 3.1 and re-run Sections 8 to 13. Default: no covariates (N_PCS_COV = 0).")

### 8.3 Covariate file for the association tests

If `N_PCS_COV > 0`, the first PCs are written as covariates in **the format each program needs** (all in the BAM-list order): a header-less numeric table for ANGSD (`-cov`), a PLINK covariate file with `FID IID PC1 ...`, and extra columns in the SNPTEST sample file (Section 11). With `N_PCS_COV = 0` nothing is written and all tests are unadjusted.

In [ ]:
COV_COLS = [f"PC{i+1}" for i in range(N_PCS_COV)]
COVFILE = D["angsd"] / "covariates.txt"
PLINK_COVFILE = D["plink"] / "covariates.txt"
if N_PCS_COV > 0:
    covdf = pcs.set_index("sampleID").loc[samples, COV_COLS]
    covdf.to_csv(COVFILE, sep="\t", header=False, index=False)
    pd.DataFrame({"FID": samples, "IID": samples}).join(covdf.reset_index(drop=True)).to_csv(PLINK_COVFILE, sep="\t", index=False)
    ANGSD_COV_ARG = f"-cov {COVFILE}"
    print(f"Using {N_PCS_COV} principal components as covariates: {COV_COLS}")
else:
    ANGSD_COV_ARG = ""
    print("No covariates: all three association tests are unadjusted.")

## 9. Association Mapping with ANGSD (`-doAsso`)

### 9.1 What ANGSD tests

ANGSD can test association **inside the same run that computes the genotype likelihoods**, so that no genotype is ever hard-called. We use the binary phenotype (`-yBin`: 1 = `m`, 0 = `i`) and the same filters as in the variant calling, so that the tested sites are the SNPs of Section 7 (minus those removed by the extra association filters).

| `-doAsso` | Test | Input | Output |
|---|---|---|---|
| **1** | **Allele-frequency likelihood-ratio test**: are the minor-allele frequencies different between cases and controls? Computed from the genotype *likelihoods*. | GL + major/minor + MAF | LRT |
| **2** | **Score test** of the genotype (as expected dosage under the posterior) in a logistic model. Very fast; supports covariates. | genotype posteriors (`-doPost 1`) | LRT |
| **4** | **Latent genotype model**: the genotype is an unobserved variable, the logistic regression is fitted by EM. Slower; also returns an effect size. | genotype posteriors | LRT, `beta`, `SE` |

The statistic `LRT` is $\chi^2$ with 1 degree of freedom under the null; we convert it to a p-value with $P = 1 - F_{\chi^2_1}(\mathrm{LRT})$.

| Flag | Meaning |
|---|---|
| `-yBin pheno.ybin` | binary phenotype, one value per line in BAM-list order (`-999` = missing) |
| `-doAsso k` | the test above |
| `-doPost 1` | posteriors with the estimated allele frequency as prior (the recommended prior; `-doPost 2`, a uniform prior, is known to inflate the p-values) |
| `-cov file` | covariates (only if `N_PCS_COV > 0`) |
| `-minHigh h` | require at least $h$ individuals with a posterior $>0.9$ in at least two of the three genotype classes. The value suggested in the ANGSD documentation (10) is **very strict for <1x data**, because confident genotypes are rare. We set it to 0 (off). |
| `-minCount c` | require an expected minor-allele count $\ge c$ (frequency $\times\, 2N$) |

> **Caveat on inflation.** The ANGSD documentation lists the typical causes of inflated p-values: a uniform prior, too-low `-minHigh`/`-minCount`, non-normal quantitative traits, population structure and small samples. We therefore check $\lambda_{GC}$ and the QQ plot below. Since we relaxed `-minHigh`, keep an eye on them.

The ANGSD result file has one row per site, **including sites where the test failed** (the statistic is set to `-999`). "Number of sites in the file" and "number of sites tested" are therefore different numbers, which will matter in the comparison.

In [ ]:
def lambda_gc(p):
    # Genomic inflation factor: median of the observed chi-square statistics / median of chi2 with 1 d.f. (0.4549)
    p = np.asarray(p, dtype=float)
    p = p[np.isfinite(p) & (p > 0) & (p <= 1)]
    return float(np.median(stats.chi2.isf(p, 1)) / stats.chi2.ppf(0.5, 1)) if len(p) else np.nan

PLOT_OFFSET, _cum = {}, 0
for c in ANALYSED:
    PLOT_OFFSET[c] = _cum; _cum += int(fai.loc[c, "length"])
PLOT_TOTAL = _cum
CHR_LABEL = fai.label.fillna("?").to_dict()

def in_peak(df):
    return (df.contig == M_CHROM) & df.pos.between(PEAK_START, PEAK_END)

def manhattan(ax, df, title, color, n_tests=None, ymax=None):
    d = df.dropna(subset=["p"]).copy()
    d = d[d.contig.isin(PLOT_OFFSET)]
    d["x"] = d.contig.map(PLOT_OFFSET) + d.pos
    d["logp"] = -np.log10(d.p.clip(lower=1e-300))
    parity = d.contig.map({c: i % 2 for i, c in enumerate(ANALYSED)})
    ax.scatter(d.x[parity == 0], d.logp[parity == 0], s=4, color=color, alpha=0.8, linewidths=0)
    ax.scatter(d.x[parity == 1], d.logp[parity == 1], s=4, color="#7f7f7f", alpha=0.8, linewidths=0)
    pk = in_peak(d)
    if pk.any():
        ax.scatter(d.x[pk], d.logp[pk], s=22, color="red", zorder=5, label="M locus window")
        ax.legend(loc="upper right", fontsize=8)
    if n_tests:
        ax.axhline(-np.log10(0.05 / n_tests), c="r", ls="--", lw=1)
    centers = [PLOT_OFFSET[c] + fai.loc[c, "length"] / 2 for c in ANALYSED]
    ax.set_xticks(centers); ax.set_xticklabels([CHR_LABEL[c] for c in ANALYSED], fontsize=7)
    ax.set_xlim(0, PLOT_TOTAL)
    if ymax: ax.set_ylim(0, ymax)
    ax.set(ylabel=r"$-\log_{10}$ P", title=title)

def qq(ax, p, color, label=None):
    p = np.sort(np.asarray(p, dtype=float)); p = p[np.isfinite(p) & (p > 0)]
    n = len(p)
    if n == 0:
        return
    exp = -np.log10((np.arange(1, n + 1) - 0.5) / n); obs = -np.log10(p)
    lam = lambda_gc(p)
    ax.scatter(exp, obs, s=6, color=color, label=(f"{label}  λ={lam:.2f}" if label else f"λ={lam:.2f}"))
    m = max(exp.max(), obs.max()) * 1.02
    ax.plot([0, m], [0, m], "k--", lw=1)
    ax.set(xlabel="expected -log10 P", ylabel="observed -log10 P"); ax.legend(fontsize=8)

def top_hits(df, n=8):
    t = df.dropna(subset=["p"]).nsmallest(n, "p")[[c for c in ["contig", "pos", "p", "maf", "beta", "info"] if c in df.columns]].copy()
    t["chrom_label"] = t.contig.map(CHR_LABEL)
    t["in_M_window"] = in_peak(t)
    return t.reset_index(drop=True)

print("helpers defined: lambda_gc, manhattan, qq, top_hits, in_peak")

In [ ]:
def read_angsd_assoc(path):
    d = pd.read_csv(path, sep="\t")
    d = d.loc[:, ~d.columns.astype(str).str.startswith("Unnamed")]
    d = d.rename(columns={"Chromosome": "contig", "Position": "pos", "Frequency": "maf", "beta": "beta", "SE": "se"})
    lrt_col = [c for c in d.columns if str(c).upper().startswith("LRT")][0]
    lrt = pd.to_numeric(d[lrt_col], errors="coerce")
    d["lrt"] = lrt
    d["tested"] = lrt.notna() & (lrt > -1)                  # failed sites carry -999
    d["p"] = np.where(d.tested, stats.chi2.sf(lrt.clip(lower=0), 1), np.nan)
    return d

asso = {}
for mode in ASSO_MODES:
    prefix = D["angsd"] / f"assoc_mode{mode}"
    if not done(f"{prefix}.lrt0.gz"):
        run(f"angsd {ANGSD_COMMON} -doPost 1 -yBin {YBIN} -doAsso {mode} -minHigh {ASSO_MIN_HIGH} "
            f"-minCount {ASSO_MIN_COUNT} {ANGSD_COV_ARG} -out {prefix}", log=D["angsd"] / f"angsd_assoc_mode{mode}.log", show=4)
    else:
        print(f"ANGSD -doAsso {mode}: output exists, skipping.")
    f = sorted(glob.glob(f"{prefix}.lrt*.gz"))
    assert f, f"No association output for mode {mode}: see angsd_assoc_mode{mode}.log"
    asso[mode] = read_angsd_assoc(f[0])

summ = pd.DataFrame([dict(mode=m, sites_in_file=len(d), sites_tested=int(d.tested.sum()),
                          min_P=d.p.min(), lambda_GC=lambda_gc(d.p)) for m, d in asso.items()])
display(summ.style.format({"min_P": "{:.2e}", "lambda_GC": "{:.3f}"}))
assert ASSO_PRIMARY in asso, "ASSO_PRIMARY must be one of ASSO_MODES"
print("\nFirst rows of the primary result (mode", ASSO_PRIMARY, "):")
display(asso[ASSO_PRIMARY].head())

### 9.2 Results: Manhattan plot, QQ plot and top hits

* **Manhattan plot**: one point per tested SNP; the x-axis is the position along the concatenated chromosomes (alternating colours), the y-axis is $-\log_{10}P$. The red dashed line is the Bonferroni threshold for the number of SNPs tested by *this* program. Points inside the **M-locus window** (chromosome 26, 6.731 to 6.743 Mb) are red.
* **QQ plot** for every ANGSD mode, with $\lambda_{GC}$.
* The **top hits** table says whether the strongest SNPs fall inside the M window.

Remember the positive-control logic: the dominant allele **M** is defined by large TE insertions, and `mm` samples do not have reads in them. Low-coverage, small-subset data may give a weaker signal than the paper's 0.81x full data. A peak that is *near* the window rather than inside it is still informative, because linkage disequilibrium extends beyond the 12 kb.

In [ ]:
prim = asso[ASSO_PRIMARY]
prim_t = prim[prim.tested].copy()
n_t = len(prim_t)
assert n_t > 0, "ANGSD tested no site: lower ASSO_MIN_COUNT / ASSO_MIN_HIGH in Section 3.1 and re-run Section 9."

fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(2, 1, 1)
manhattan(ax1, prim_t, f"ANGSD -doAsso {ASSO_PRIMARY}: {n_t:,} SNPs tested", TOOL_COLORS["ANGSD"], n_tests=n_t)
ax2 = fig.add_subplot(2, 2, 3)
for m_, d in asso.items():
    qq(ax2, d.p, color=plt.cm.tab10(list(asso).index(m_)), label=f"mode {m_}")
ax2.set_title("QQ plot, all ANGSD modes")
ax3 = fig.add_subplot(2, 2, 4)
if len(asso) > 1:
    other = [m_ for m_ in asso if m_ != ASSO_PRIMARY]
    for m_ in other:
        j = prim_t.merge(asso[m_][asso[m_].tested], on=["contig", "pos"], suffixes=("", "_o"))
        ax3.scatter(-np.log10(j.p.clip(lower=1e-300)), -np.log10(j.p_o.clip(lower=1e-300)), s=5, alpha=0.5, label=f"mode {m_}")
        rho = stats.spearmanr(j.p, j.p_o)[0]
        print(f"Spearman correlation of P between mode {ASSO_PRIMARY} and mode {m_}: {rho:.3f} ({len(j):,} shared SNPs)")
    ax3.set(xlabel=f"-log10 P, mode {ASSO_PRIMARY}", ylabel="-log10 P, other mode", title="Agreement between ANGSD modes"); ax3.legend()
plt.tight_layout(); plt.show()

print("Top hits of the primary ANGSD test:")
display(top_hits(prim_t))

## 10. Association Mapping with PLINK (hard-called genotypes)

### 10.1 The route: posteriors, hard calls, PLINK

PLINK needs **one genotype per individual and variant**. We therefore reduce the ANGSD posteriors to hard calls with the rule of Section 0.2:

$$\hat g = \arg\max_g P(G=g\mid D)\quad\text{if}\quad \max_g P(G=g\mid D)\ \ge\ \texttt{POST\_CUTOFF},\qquad \text{otherwise missing}.$$

The genotypes are written as a VCF with only the `GT` field (`0/0`, `0/1`, `1/1`, `./.`), which PLINK imports with `--vcf`. In real life this step is usually done by a variant caller such as `bcftools call` or GATK; here we use the ANGSD posteriors so that the three programs start from **the same evidence** and differ only in how they treat uncertainty.

### 10.2 PLINK in three minutes

PLINK stores genotypes in three files, the **binary fileset**:

| File | Content |
|---|---|
| `.bed` | genotypes, 2 bits each |
| `.bim` | one line per variant: chromosome, ID, genetic distance, position, allele 1, allele 2 |
| `.fam` | one line per individual: family ID, individual ID, father, mother, sex, **phenotype** |

PLINK codes a binary phenotype as **1 = control, 2 = case** (and 0 / -9 = missing). We therefore write `i` as 1 and `m` as 2.

Key options:

| Option | Meaning |
|---|---|
| `--vcf f --double-id` | import a VCF; family ID = individual ID = sample name |
| `--allow-extra-chr` | accept contig names that are not human chromosomes 1-22, X, Y (all of ours!) |
| `--set-missing-var-ids @:#` | give every variant the ID `contig:position` (ANGSD leaves it empty) |
| `--keep-allele-order` | keep `REF`/`ALT` as in the VCF (otherwise PLINK relabels A1 as the minor allele) |
| `--pheno` | external phenotype file |
| `--geno g` | remove variants with more than a fraction `g` of **missing** genotypes |
| `--maf m` | remove variants with minor-allele frequency below `m` (among called genotypes) |
| `--hwe p` | remove variants whose Hardy-Weinberg exact-test P is below `p` (**controls only** when cases and controls are present) |
| `--assoc` | allelic chi-square test, 1 d.f. (`--logistic` if covariates are used) |
| `--test-missing` | test whether the missingness of each variant differs between cases and controls |

The standard "textbook" thresholds (`--geno 0.02`, i.e. 98% call rate) would remove essentially **everything** at low coverage, because most genotypes are missing. We use `PLINK_GENO = 0.95`: a variant is kept if at least 5% of the individuals have a called genotype. If even that leaves too few variants, the notebook relaxes the threshold automatically (0.98, then 0.995) and tells you.

In [ ]:
def write_hardcalls(src_vcf, dst_vcf, cutoff):
    # Hard-call genotypes from the GP field of a VCF; write a GT-only VCF (bgzipped) and return summary counts.
    plain = Path(str(dst_vcf)[:-3])
    GT = np.array(["0/0", "0/1", "1/1", "./."])
    n_sites = n_called = 0
    with open(plain, "w") as fh:
        fh.write("##fileformat=VCFv4.2\n")
        fh.write('##FORMAT=<ID=GT,Number=1,Type=String,Description="Hard call from the ANGSD posterior (missing if max posterior < cutoff)">\n')
        fh.write("".join(f"##contig=<ID={c},length={l}>\n" for c, l in zip(fai.contig, fai.length)))
        fh.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\t" + "\t".join(samples) + "\n")
        for chrom, pos, ref, alt, gp in iter_gp(src_vcf):
            conf = gp.max(axis=1)
            idx = np.where(conf >= cutoff, gp.argmax(axis=1), 3)
            n_sites += 1; n_called += int((idx < 3).sum())
            fh.write(f"{chrom}\t{pos}\t.\t{ref}\t{alt}\t.\tPASS\t.\tGT\t" + "\t".join(GT[idx]) + "\n")
    run(f"bgzip -f {plain} && bcftools index -t -f {dst_vcf}", quiet=True)
    return n_sites, n_called

HARD_VCF = D["plink"] / "hardcalls.vcf.gz"
if not done(HARD_VCF, str(HARD_VCF) + ".tbi"):
    t0 = time.time()
    n_s, n_c = write_hardcalls(CALLS_VCF, HARD_VCF, POST_CUTOFF)
    print(f"Hard calls written in {time.time() - t0:.0f} s: {n_s:,} variants x {N} individuals; "
          f"{100*n_c/(n_s*N):.1f} % of the genotypes were called, {100 - 100*n_c/(n_s*N):.1f} % are missing")
else:
    print("Hard-call VCF already exists:", HARD_VCF.name)
print(run(f"bcftools view -H {HARD_VCF} | head -n 3 | cut -f 1-14", quiet=True).stdout)

In [ ]:
# ---- phenotype file for PLINK: FID IID PHENO  (1 = control, 2 = case)
PLINK_PHENO = D["plink"] / "pheno.txt"
pd.DataFrame({"FID": samples, "IID": samples, "P": pheno.y.map({0: 1, 1: 2})}).to_csv(PLINK_PHENO, sep="\t", header=False, index=False)

# ---- import to a binary fileset
RAW = D["plink"] / "raw"
if not done(f"{RAW}.bed", f"{RAW}.bim", f"{RAW}.fam"):
    run(f"plink --vcf {HARD_VCF} --double-id --allow-extra-chr --set-missing-var-ids @:# --keep-allele-order "
        f"--make-bed --out {RAW} --threads {THREADS}", log=D["plink"] / "plink_import.log", show=6)
def n_bim(prefix): return sum(1 for _ in open(f"{prefix}.bim"))
plink_counts = {"hard-called variants (PLINK import)": n_bim(RAW)}
print("Variants imported:", plink_counts)

### 10.3 Quality control, one filter at a time

We apply the filters in three separate PLINK calls, so that we can count how many variants each of them removes. (PLINK would do the same in one command, but then reports less clearly what happened.) The genotype call rate filter comes first because the Hardy-Weinberg test and the allele frequency are only meaningful when enough genotypes exist.

In [ ]:
Q1, Q2, Q3 = D["plink"] / "qc1_geno", D["plink"] / "qc2_maf", D["plink"] / "qc3_hwe"

# step 1: call-rate filter. It is cheap, so we always redo it, and relax the threshold if (almost) nothing survives.
geno_used = None
for g in sorted(set([PLINK_GENO, 0.98, 0.995])):
    if g < PLINK_GENO: continue
    run(f"plink --bfile {RAW} --allow-extra-chr --pheno {PLINK_PHENO} --geno {g} --keep-allele-order "
        f"--make-bed --out {Q1} --threads {THREADS}", log=D["plink"] / f"{Q1.name}.log", quiet=True)
    if n_bim(Q1) >= 50:
        geno_used = g; break
if geno_used is None:
    geno_used = g
elif geno_used != PLINK_GENO:
    print(f"NOTE: --geno {PLINK_GENO} left fewer than 50 variants; relaxed automatically to --geno {geno_used}.")
plink_counts[f"after --geno {geno_used}"] = n_bim(Q1)

steps = [(Q1, Q2, f"--maf {PLINK_MAF}", f"after --maf {PLINK_MAF}"),
         (Q2, Q3, f"--hwe {PLINK_HWE}", f"after --hwe {PLINK_HWE} (controls)")]
for src, dst, flt, label in steps:                       # cheap, so always redone
    run(f"plink --bfile {src} --allow-extra-chr --pheno {PLINK_PHENO} {flt} --keep-allele-order "
        f"--make-bed --out {dst} --threads {THREADS}", log=D["plink"] / f"{dst.name}.log", quiet=True)
    plink_counts[label] = n_bim(dst)
QC = Q3
cnt = pd.Series(plink_counts, name="variants")
display(pd.DataFrame({"variants": cnt, "removed_by_step": (-cnt.diff()).fillna(0).astype(int)}))
assert cnt.iloc[-1] > 0, ("No variant survived the PLINK QC. The hard calls are too sparse: lower POST_CUTOFF (e.g. 0.8), "
                          "raise PLINK_GENO (e.g. 0.995) or relax PLINK_MAF / PLINK_HWE in Section 3.1 and re-run from Section 10.")

### 10.4 The association test, and a test for differential missingness

`plink --assoc` builds, for each variant, the 2x2 table of allele counts in cases and controls and computes the Pearson chi-square test (1 d.f.). It reports the allele frequency in cases (`F_A`) and controls (`F_U`), the odds ratio, and the p-value. With covariates (`N_PCS_COV > 0`) we switch to `--logistic` and keep only the `ADD` (additive genotype) rows.

We also run `--test-missing`. For each variant it tests whether the **fraction of missing genotypes differs between cases and controls**. This is the hard-call counterpart of the TE-insertion problem: where `mm` (control) samples have no reads, their genotypes are missing and the missingness itself is associated with the phenotype. A strong signal here **near the M locus** is expected and is a feature of the biology, but elsewhere in the genome it would indicate a technical problem.

In [ ]:
ASSOC = D["plink"] / "assoc"
if N_PCS_COV > 0:
    assoc_opts = f"--logistic hide-covar --covar {PLINK_COVFILE} --covar-name {','.join(COV_COLS)}"
    assoc_file = f"{ASSOC}.assoc.logistic"
else:
    assoc_opts = "--assoc"
    assoc_file = f"{ASSOC}.assoc"
if not done(assoc_file):
    run(f"plink --bfile {QC} --allow-extra-chr --pheno {PLINK_PHENO} {assoc_opts} --out {ASSOC} --threads {THREADS}",
        log=D["plink"] / "plink_assoc.log", show=4)
FREQ = D["plink"] / "freq"
if not done(f"{FREQ}.frq"):
    run(f"plink --bfile {QC} --allow-extra-chr --freq --out {FREQ}", quiet=True)
MISS = D["plink"] / "missingness_test"
if not done(f"{MISS}.missing"):
    run(f"plink --bfile {RAW} --allow-extra-chr --pheno {PLINK_PHENO} --test-missing --out {MISS}", quiet=True)

def split_id(ids):
    parts = ids.str.rsplit(":", n=1, expand=True)
    return parts[0], parts[1].astype(int)

pl = pd.read_csv(assoc_file, sep=r"\s+")
if "TEST" in pl.columns:
    pl = pl[pl.TEST == "ADD"]
pl["contig"], pl["pos"] = split_id(pl.SNP)
pl["p"] = pd.to_numeric(pl.P, errors="coerce")
fq = pd.read_csv(f"{FREQ}.frq", sep=r"\s+")
fq["contig"], fq["pos"] = split_id(fq.SNP)
fq["maf"] = pd.to_numeric(fq.MAF, errors="coerce").pipe(lambda m: np.minimum(m, 1 - m))     # fold: A1 need not be the minor allele
pl = pl.merge(fq[["contig", "pos", "maf", "NCHROBS"]], on=["contig", "pos"], how="left")
pl["n_called"] = pl.NCHROBS / 2
if "OR" in pl.columns: pl["or_"] = pd.to_numeric(pl.OR, errors="coerce")
plink_res = pl.reset_index(drop=True)

ms = pd.read_csv(f"{MISS}.missing", sep=r"\s+")
ms["contig"], ms["pos"] = split_id(ms.SNP)
ms["p"] = pd.to_numeric(ms.P, errors="coerce")

print(f"PLINK: {len(plink_res):,} variants in the result file, {int(plink_res.p.notna().sum()):,} with a valid P; "
      f"lambda_GC = {lambda_gc(plink_res.p):.3f}")
print(f"Individuals with a called genotype per variant: median {plink_res.n_called.median():.0f} of {N}")
display(plink_res[[c for c in ["contig", "pos", "A1", "A2", "F_A", "F_U", "maf", "n_called", "p", "or_"] if c in plink_res.columns]].head())

In [ ]:
fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(2, 1, 1)
pl_t = plink_res.dropna(subset=["p"])
manhattan(ax1, pl_t, f"PLINK {'--assoc' if N_PCS_COV == 0 else '--logistic'} on hard calls: {len(pl_t):,} variants tested", TOOL_COLORS["PLINK"], n_tests=len(pl_t))
ax2 = fig.add_subplot(2, 2, 3); qq(ax2, pl_t.p, TOOL_COLORS["PLINK"], "PLINK"); ax2.set_title("QQ plot")
ax3 = fig.add_subplot(2, 2, 4)
mm_ = ms.dropna(subset=["p"])
ax3.scatter(mm_.pos[mm_.contig == M_CHROM] / 1e6, -np.log10(mm_.p[mm_.contig == M_CHROM].clip(lower=1e-300)), s=5, color="grey")
ax3.axvspan(PEAK_START / 1e6, PEAK_END / 1e6, color="red", alpha=0.2, label="M locus window")
ax3.set(xlabel=f"position on {M_CHROM} (Mb)", ylabel="-log10 P (missingness differs m vs i)", title="Differential missingness along the M chromosome")
ax3.legend(); plt.tight_layout(); plt.show()
print("Top hits of PLINK:"); display(top_hits(pl_t))
print(f"Genome-wide lambda of the missingness test: {lambda_gc(mm_.p):.2f} "
      "(a value far above 1 would mean that cases and controls differ in coverage everywhere: a technical confounder)")

## 11. Association Mapping with SNPTEST (genotype probabilities)

### 11.1 SNPTEST and the Oxford formats

SNPTEST (Marchini et al. 2007; Marchini & Howie 2010) is the classic program of the IMPUTE/WTCCC family for single-variant tests. Its model is built for **imputed** data, which, like ours, come as three probabilities per genotype rather than a single call. Its two input files:

**`.gen` (genotype probabilities).** One line per variant, then three numbers per individual:

```
chromosome  SNPID  rsID  position  alleleA  alleleB  P(AA)_1 P(AB)_1 P(BB)_1  P(AA)_2 P(AB)_2 P(BB)_2 ...
```

**`.sample`**. A table with a header line and a second line giving the *type* of each column (`0` = ID column, `B` = binary phenotype / covariate `C` = continuous covariate):

```
ID_1      ID_2      missing  morph01  PC1
0         0         0        B        C
CAM035143 CAM035143 0        1        -0.021
...
```

Two practical conversions are needed because our genome is not human:

1. **Chromosome and position.** SNPTEST expects a chromosome label such as `01`. Butterfly contig names (`OZxxxxxx.1`) are not accepted by all versions, and positions must increase. We therefore write every variant with chromosome `01`, and encode the genome position as a **cumulative coordinate** (chromosome offset plus position). The real `contig:position` is stored in the variant ID, from which we read it back after the analysis.
2. **Probabilities.** We use the ANGSD posteriors `GP` (REF = major allele = allele A, ALT = minor = allele B), renormalised to sum to 1 and written with 5 decimals.

### 11.2 The command

```
snptest -data x.gen x.sample -frequentist 1 -method score -pheno morph01 -o out.txt
```

| Option | Meaning |
|---|---|
| `-data gen sample` | input files |
| `-frequentist 1` | **additive** genotype model (1 = additive, 2 = dominant, 3 = recessive, 4 = general, 5 = heterozygote) |
| `-method score` | the **missing-data score test**: valid for binary traits, accounts for genotype uncertainty, and falls back to EM when it cannot be computed. Alternatives: `newml` (maximum likelihood, gives `beta` and `se`), `threshold` (uses hard calls at `-call_thresh`, default 0.9), `expected` (dosages). Note that `em` is for continuous traits only. |
| `-pheno morph01` | the phenotype column of the sample file |
| `-cov_names PC1 PC2` | covariates (only if `N_PCS_COV > 0`) |
| `-o` | output file |

The output has one row per variant. The columns that matter to us:

| Column | Meaning |
|---|---|
| `alternate_ids`, `rsid`, `chromosome`, `position` | our `contig:position` identifiers |
| `alleleA`, `alleleB` | allele A (major) and B (minor) |
| `all_maf`, `cases_maf`, `controls_maf` | minor-allele frequency in everyone and by group |
| `missing_data_proportion` | fraction of individuals without data |
| `info` | the **IMPUTE information measure** ($0$ = genotypes unknown, $1$ = known), computed from the probabilities |
| `frequentist_add_pvalue` | P of the additive-model test |
| `frequentist_add_beta_1`, `frequentist_add_se_1` | log-odds ratio per copy of allele B and its standard error (not for every method) |

The **`info` score** is exactly the quantity we need to discuss low coverage: it is low when most individuals have posteriors close to the prior. A threshold such as 0.3 to 0.4 is standard for imputed GWAS; at <1x it filters a great deal. We use `SNPTEST_MIN_INFO` (0.1 by default) when building VCF 3.

In [ ]:
GEN = D["snptest"] / "hypmis.gen"
SAMPLEF = D["snptest"] / "hypmis.sample"
SNPTEST_OUT = D["snptest"] / "snptest_results.txt"

def write_gen(src_vcf, gen_path):
    fmt = " ".join(["%.5f"] * (3 * N))
    tmp = WORK / "hypmis_unsorted.gen"; n = 0
    with open(tmp, "w") as fh:
        for chrom, pos, ref, alt, gp in iter_gp(src_vcf):
            g = np.nan_to_num(gp.astype(np.float64), nan=0.0)
            s = g.sum(axis=1, keepdims=True)
            g = np.where(s > 0, g / np.where(s > 0, s, 1.0), 0.0)
            snp = f"{chrom}:{pos}"
            fh.write(f"01 {snp} {snp} {OFFSET[chrom] + pos} {ref} {alt} " + (fmt % tuple(g.ravel().tolist())) + "\n")
            n += 1
    run(f"LC_ALL=C sort -t ' ' -k4,4n -s {tmp} > {gen_path}", quiet=True)      # SNPTEST likes increasing positions
    return n

if not done(GEN):
    t0 = time.time(); n_gen = write_gen(CALLS_VCF, GEN)
    print(f"{n_gen:,} variants written to {GEN.name} in {time.time() - t0:.0f} s")
else:
    print("GEN file already exists:", GEN.name)

# ---- sample file: a header, a type line, then one row per individual (same order as the VCF)
cols = ["ID_1", "ID_2", "missing", "morph01"] + (COV_COLS if N_PCS_COV > 0 else [])
types = ["0", "0", "0", "B"] + (["C"] * N_PCS_COV)
body = pd.DataFrame({"ID_1": samples, "ID_2": samples, "missing": 0, "morph01": pheno.y.values})
if N_PCS_COV > 0:
    body = body.join(pcs.set_index("sampleID").loc[samples, COV_COLS].reset_index(drop=True))
with open(SAMPLEF, "w") as fh:
    fh.write(" ".join(cols) + "\n" + " ".join(types) + "\n")
    body.to_csv(fh, sep=" ", header=False, index=False)
print("\nFirst lines of the .sample file:"); print(run(f"head -n 4 {SAMPLEF}", quiet=True).stdout)
print("First variant of the .gen file (first 12 fields):"); print(run(f"head -n 1 {GEN} | cut -d' ' -f1-12", quiet=True).stdout)

In [ ]:
assert shutil.which("snptest"), "SNPTEST is not installed: see Section 2 or set SNPTEST_BINARY in Section 3.1."
if not done(SNPTEST_OUT):
    cov_arg = f"-cov_names {' '.join(COV_COLS)}" if N_PCS_COV > 0 else ""
    run(f"snptest -data {GEN} {SAMPLEF} -frequentist 1 -method {SNPTEST_METHOD} -pheno morph01 {cov_arg} "
        f"-o {SNPTEST_OUT} -log {D['snptest'] / 'snptest.log'}", log=D["snptest"] / "snptest_stdout.log", show=8)
else:
    print("SNPTEST output already exists:", SNPTEST_OUT.name)

sn = pd.read_csv(SNPTEST_OUT, sep=r"\s+", comment="#")
pcol = next(c for c in sn.columns if c.endswith("add_pvalue") or c.endswith("_pvalue"))
print("SNPTEST columns:", list(sn.columns))
print("p-value column used:", pcol)
sn["contig"], sn["pos"] = split_id(sn.rsid.astype(str))
p_raw = pd.to_numeric(sn[pcol], errors="coerce")
sn["p"] = p_raw.where((p_raw >= 0) & (p_raw <= 1)).clip(lower=1e-300)
sn["maf"] = pd.to_numeric(sn["all_maf"], errors="coerce").pipe(lambda m: np.minimum(m, 1 - m)) if "all_maf" in sn.columns else np.nan
sn["info"] = pd.to_numeric(sn["info"], errors="coerce") if "info" in sn.columns else np.nan
bcol = [c for c in sn.columns if c.endswith("add_beta_1")]
sn["beta"] = pd.to_numeric(sn[bcol[0]], errors="coerce") if bcol else np.nan
snptest_all = sn.reset_index(drop=True)
print(f"\nSNPTEST: {len(snptest_all):,} variants in the output, {int(snptest_all.p.notna().sum()):,} with a valid P; "
      f"lambda_GC = {lambda_gc(snptest_all.p):.3f}")
display(snptest_all[["contig", "pos", "alleleA", "alleleB", "maf", "info", "p", "beta"]].head())

In [ ]:
fig = plt.figure(figsize=(16, 8))
sn_t = snptest_all[(snptest_all.p.notna()) & (snptest_all["info"].fillna(0) >= SNPTEST_MIN_INFO)]
assert len(sn_t) > 0, f"No SNPTEST variant has info >= {SNPTEST_MIN_INFO}: lower SNPTEST_MIN_INFO in Section 3.1 and re-run."
ax1 = fig.add_subplot(2, 1, 1)
manhattan(ax1, sn_t, f"SNPTEST -method {SNPTEST_METHOD}: {len(sn_t):,} variants with info >= {SNPTEST_MIN_INFO}", TOOL_COLORS["SNPTEST"], n_tests=len(sn_t))
ax2 = fig.add_subplot(2, 3, 4); qq(ax2, sn_t.p, TOOL_COLORS["SNPTEST"], "SNPTEST"); ax2.set_title("QQ plot")
ax3 = fig.add_subplot(2, 3, 5)
ax3.hist(snptest_all["info"].dropna(), bins=50, color="grey"); ax3.axvline(SNPTEST_MIN_INFO, c="r", ls="--", label=f"threshold {SNPTEST_MIN_INFO}")
ax3.set(xlabel="SNPTEST info", ylabel="variants", title="Information content of the genotypes"); ax3.legend()
ax4 = fig.add_subplot(2, 3, 6)
ax4.scatter(snptest_all.maf, snptest_all["info"], s=4, alpha=0.4, color="grey")
ax4.set(xlabel="minor-allele frequency", ylabel="info", title="Info is higher for common variants")
plt.tight_layout(); plt.show()

thr = pd.DataFrame({"min_info": [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.8]})
thr["variants_kept"] = [int(((snptest_all["info"].fillna(0) >= t) & snptest_all.p.notna()).sum()) for t in thr.min_info]
display(thr.T)
print("Top hits of SNPTEST (info filter applied):"); display(top_hits(sn_t))

### 11.3 Collecting the three result tables

From now on every result is a table with the same columns (`contig`, `pos`, `p`, `maf`, ...), which makes the comparison simple. For each program we keep **exactly the variants for which a valid test exists** (and that pass the program's own filters): these are the variants that will be written to the three VCF files.

In [ ]:
ang = prim_t.copy()
ang["beta"] = np.nan
if 4 in asso and "beta" in asso[4].columns:
    ang = ang.merge(asso[4][["contig", "pos", "beta"]].rename(columns={"beta": "beta4"}), on=["contig", "pos"], how="left")
    ang["beta"] = ang["beta4"]
ang = ang[[c for c in ["contig", "pos", "Major", "Minor", "maf", "lrt", "p", "beta"] if c in ang.columns]]
res = {
    "ANGSD": ang.reset_index(drop=True),
    "PLINK": plink_res.dropna(subset=["p"])[[c for c in ["contig", "pos", "A1", "A2", "maf", "n_called", "p", "or_"] if c in plink_res.columns]].reset_index(drop=True),
    "SNPTEST": sn_t[["contig", "pos", "alleleA", "alleleB", "maf", "info", "p", "beta"]].reset_index(drop=True),
}
for k, v in res.items():
    v["key"] = v.contig + ":" + v.pos.astype(str)
    print(f"{k:8s}: {len(v):>8,} tested variants   lambda_GC = {lambda_gc(v.p):.3f}   "
          f"smallest P = {v.p.min():.2e}   significant at Bonferroni (0.05/{len(v):,}): {int((v.p < 0.05/len(v)).sum())}")

## 12. Writing the Three VCF Files

### 12.1 Strategy

For each program we write a VCF that contains **exactly the variants the program tested** (those in `res[...]`, Section 11.3) and stores its statistics in the `INFO` column. We use BCFtools:

| Step | Command | What it does |
|---|---|---|
| select sites | `bcftools view -T targets.tsv.gz in.vcf.gz` | keep only the listed `CHROM POS` (a bgzipped, tabix-indexed two-column file) |
| annotate | `bcftools annotate -a annot.tsv.gz -h header.txt -c CHROM,POS,INFO/TAG,...` | copy the statistics from a tab-separated file into `INFO`; the header file declares the new tags |
| index | `bcftools index -t` | tabix index, required for random access |

The three **genotype sources** differ, and that is intentional:

| VCF | Source VCF | `FORMAT` fields | `INFO` tags added |
|---|---|---|---|
| **1. ANGSD** | the ANGSD calls (`HypMis_ANGSD_calls.vcf.gz`) | `GP`, `GL` (posteriors, likelihoods) | `ANGSD_LRT`, `ANGSD_P`, `ANGSD_MAF` |
| **2. PLINK** | PLINK's own export of the QC-filtered hard calls | `GT` only (many `./.`) | `PLINK_P`, `PLINK_MAF`, `PLINK_NCALLED`, `PLINK_OR` |
| **3. SNPTEST** | the ANGSD calls | `GP`, `GL` | `SNPTEST_P`, `SNPTEST_INFO`, `SNPTEST_MAF`, `SNPTEST_BETA` |

First we let PLINK export its filtered fileset as a VCF (`--recode vcf-iid bgz`; `vcf-iid` uses the sample ID alone as the sample name).

In [ ]:
PLINK_FINAL = D["plink"] / "qc_final"
if not done(f"{PLINK_FINAL}.vcf.gz"):
    run(f"plink --bfile {QC} --allow-extra-chr --keep-allele-order --recode vcf-iid bgz --out {PLINK_FINAL}",
        log=D["plink"] / "plink_export_vcf.log", quiet=True)
# make sure the header lists the contigs (PLINK may omit them for non-human chromosome names)
n_contig_lines = int(run(f"bcftools view -h {PLINK_FINAL}.vcf.gz | grep -c '^##contig' || true", quiet=True).stdout.strip() or 0)
if n_contig_lines == 0:
    run(f"bcftools annotate -h {CONTIG_HDR} -Oz -o {PLINK_FINAL}.fixed.vcf.gz {PLINK_FINAL}.vcf.gz && mv {PLINK_FINAL}.fixed.vcf.gz {PLINK_FINAL}.vcf.gz", quiet=True)
run(f"bcftools index -t -f {PLINK_FINAL}.vcf.gz", quiet=True)
print(run(f"bcftools view -H {PLINK_FINAL}.vcf.gz | wc -l", quiet=True).stdout.strip(), "variants in PLINK's exported VCF")
print(run(f"bcftools view -H {PLINK_FINAL}.vcf.gz | head -n 2 | cut -f 1-12", quiet=True).stdout)

In [ ]:
def vcf_sites(path):
    r = run(f"bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' {path}", quiet=True)
    return pd.read_csv(io.StringIO(r.stdout), sep="\t", header=None, names=["contig", "pos", "ref", "alt"],
                       dtype={"contig": str, "pos": int, "ref": str, "alt": str})

def make_vcf(tag, src_vcf, table, annots, out_vcf):
    # tag: short name for temporary files; table: DataFrame with contig, pos and the annotated columns;
    # annots: list of (column, INFO tag, VCF type, description)
    order = vcf_sites(src_vcf)[["contig", "pos"]]
    keep = order.merge(table, on=["contig", "pos"], how="inner").drop_duplicates(["contig", "pos"])     # keeps the order of the VCF
    tg, an, hd = WORK / f"{tag}_targets.tsv", WORK / f"{tag}_annot.tsv", WORK / f"{tag}_hdr.txt"
    keep[["contig", "pos"]].to_csv(tg, sep="\t", header=False, index=False)
    keep[["contig", "pos"] + [a[0] for a in annots]].to_csv(an, sep="\t", header=False, index=False, float_format="%.6g", na_rep=".")
    lines = "".join(f'##INFO=<ID={t},Number=1,Type={ty},Description="{d}">\n' for _, t, ty, d in annots)
    has_contigs = int(run(f"bcftools view -h {src_vcf} | grep -c '^##contig' || true", quiet=True).stdout.strip() or 0)
    if has_contigs == 0:
        lines += "".join(f"##contig=<ID={c},length={l}>\n" for c, l in zip(fai.contig, fai.length))
    hd.write_text(lines)
    for f in (tg, an):
        run(f"bgzip -f {f} && tabix -f -s1 -b2 -e2 {f}.gz", quiet=True)
    cols = "CHROM,POS," + ",".join(f"INFO/{t}" for _, t, _, _ in annots)
    run(f"bcftools view -T {tg}.gz {src_vcf} -Ou | bcftools annotate -a {an}.gz -h {hd} -c {cols} -Oz -o {out_vcf}", quiet=True)
    run(f"bcftools index -t -f {out_vcf}", quiet=True)
    return len(keep), len(table.drop_duplicates(["contig", "pos"]))

VCF_OUT = {
    "ANGSD":   D["vcf"] / "Hypolimnas_misippus_1_ANGSD_assoc.vcf.gz",
    "PLINK":   D["vcf"] / "Hypolimnas_misippus_2_PLINK_assoc.vcf.gz",
    "SNPTEST": D["vcf"] / "Hypolimnas_misippus_3_SNPTEST_assoc.vcf.gz",
}
SPEC = {
    "ANGSD": (CALLS_VCF, res["ANGSD"], [
        ("lrt", "ANGSD_LRT", "Float", "ANGSD doAsso score-test statistic (chi-square, 1 d.f.)"),
        ("p", "ANGSD_P", "Float", "ANGSD doAsso P-value (m vs i)"),
        ("maf", "ANGSD_MAF", "Float", "ANGSD minor allele frequency")]),
    "PLINK": (Path(f"{PLINK_FINAL}.vcf.gz"), res["PLINK"], [
        ("p", "PLINK_P", "Float", "PLINK allelic test P-value on hard calls (m vs i)"),
        ("maf", "PLINK_MAF", "Float", "PLINK minor allele frequency among called genotypes"),
        ("n_called", "PLINK_NCALLED", "Float", "Number of individuals with a hard call")]
        + ([("or_", "PLINK_OR", "Float", "PLINK odds ratio for allele A1")] if "or_" in res["PLINK"].columns else [])),
    "SNPTEST": (CALLS_VCF, res["SNPTEST"], [
        ("p", "SNPTEST_P", "Float", "SNPTEST additive-model P-value (m vs i)"),
        ("info", "SNPTEST_INFO", "Float", "SNPTEST information measure of the genotype probabilities"),
        ("maf", "SNPTEST_MAF", "Float", "SNPTEST minor allele frequency"),
        ("beta", "SNPTEST_BETA", "Float", "SNPTEST log odds ratio per copy of allele B (if available)")]),
}
for name, (src, table, annots) in SPEC.items():
    annots = [a for a in annots if a[0] in table.columns and table[a[0]].notna().any()]
    n_written, n_table = make_vcf(name, src, table, annots, VCF_OUT[name])
    flag = "" if n_written == n_table else f"   <-- {n_table - n_written:,} tested variants are NOT in the source VCF"
    print(f"{name:8s}: {n_written:>8,} variants -> {VCF_OUT[name].name}{flag}")

print()
for name, p in VCF_OUT.items():
    print(f"{p.name:55s} {p.stat().st_size/1e6:8.2f} MB   samples: {run(f'bcftools query -l {p} | wc -l', quiet=True).stdout.strip()}")
print("\nExample record of VCF 3 (INFO shows the SNPTEST tags; first 8 columns):")
print(run(f"bcftools view -H {VCF_OUT['SNPTEST']} | head -n 2 | cut -f 1-8", quiet=True).stdout)

## 13. Comparing the Three VCF Files

All comparisons below are made from the **VCF files themselves** (read with `bcftools query`), the way a collaborator receiving the three files would do it. We ask four questions:

1. **How many variants** does each file contain, and how do they relate to the ANGSD "universe"?
2. **Where** are they? Do the three files cover the same regions of the genome, and the M locus in particular?
3. Do they **agree on the alleles** (`REF`/`ALT`) and **allele frequencies**?
4. Do they **agree on the association signal**?

### 13.1 Number of variants

In [ ]:
U = vcf_sites(CALLS_VCF)
S = {k: vcf_sites(p) for k, p in VCF_OUT.items()}
for d in [U] + list(S.values()):
    d["key"] = d.contig + ":" + d.pos.astype(str)
sets = {k: set(v.key) for k, v in S.items()}
universe = set(U.key)
names = list(S)

def describe_sites(df, label):
    tr = (df.ref + df.alt).isin(["AG", "GA", "CT", "TC"]).sum()
    gaps = df.sort_values(["contig", "pos"]).groupby("contig").pos.diff().dropna()
    return dict(file=label, variants=len(df), pct_of_universe=100 * len(df) / len(U),
                on_M_chromosome=int((df.contig == M_CHROM).sum()), in_M_window=int(in_peak(df).sum()),
                chromosomes=df.contig.nunique(), Ti_Tv=tr / max(len(df) - tr, 1),
                median_gap_bp=float(gaps.median()) if len(gaps) else np.nan,
                mean_density_per_Mb=len(df) / (fai.loc[ANALYSED, "length"].sum() / 1e6))
tab = pd.DataFrame([describe_sites(U, "ANGSD calls (universe)")] + [describe_sites(S[k], f"VCF {i+1}: {k}") for i, k in enumerate(names)])
display(tab.style.format({"pct_of_universe": "{:.1f}", "Ti_Tv": "{:.2f}", "median_gap_bp": "{:,.0f}", "mean_density_per_Mb": "{:.1f}",
                          "variants": "{:,}", "on_M_chromosome": "{:,}", "in_M_window": "{:,}"}))
for k in names:
    outside = len(sets[k] - universe)
    if outside: print(f"NOTE: {k}: {outside:,} variants are not in the ANGSD universe (the association run produced a site the calling run did not).")

**Reading the table.**

* `pct_of_universe` says how much of the ANGSD call set each program kept. We expect **ANGSD and SNPTEST to keep most variants** (they use the full posteriors), and **PLINK to keep much fewer**, because every variant must have enough hard-called genotypes and pass the allele-frequency and Hardy-Weinberg filters.
* `Ti_Tv` is the transition/transversion ratio. In real polymorphism it is usually clearly above 0.5, the value expected for random errors. A value close to 0.5 indicates that many variants are artefacts.
* `median_gap_bp` and `mean_density_per_Mb` tell us how densely the variants cover the genome. A GWAS can only detect associations near typed variants, so a sparse file means low resolution.

### 13.2 Which variants are shared? An UpSet-style intersection plot

Each bar is a **non-overlapping combination** of the three files: the bar over a single dot counts the variants found *only* in that file; the bar over three dots counts the variants in all three.

In [ ]:
def upset(sets_dict, order, colors, title=""):
    allk = set().union(*sets_dict.values())
    combos = {}
    for k in allk:
        key = tuple(k in sets_dict[n] for n in order)
        combos[key] = combos.get(key, 0) + 1
    items = sorted(combos.items(), key=lambda kv: -kv[1])
    fig, (axb, axd) = plt.subplots(2, 1, figsize=(max(6, 1.3 * len(items)), 5.5), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1.2], "hspace": 0.05})
    for i, (key, v) in enumerate(items):
        axb.bar(i, v, color="#444444"); axb.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
        idx = [j for j, f in enumerate(key) if f]
        axd.plot([i, i], [min(idx), max(idx)], "k-", lw=2)
        for j, f in enumerate(key):
            axd.scatter(i, j, s=70, color=(colors[order[j]] if f else "#dddddd"), zorder=3)
    axd.set_yticks(range(len(order))); axd.set_yticklabels(order); axd.set_ylim(-0.5, len(order) - 0.5); axd.invert_yaxis()
    axd.set_xticks([]); axb.set_ylabel("variants"); axb.set_title(title)
    for ax in (axb, axd):
        for s in ("top", "right"): ax.spines[s].set_visible(False)
    plt.show()
    return pd.DataFrame([{**{n: f for n, f in zip(order, key)}, "variants": v} for key, v in items])

inter = upset(sets, names, TOOL_COLORS, "Intersections of the variant sets in the three VCF files")
display(inter)

J = pd.DataFrame(index=names, columns=names, dtype=float); Cn = J.copy()
for a in names:
    for b in names:
        J.loc[a, b] = len(sets[a] & sets[b]) / max(len(sets[a] | sets[b]), 1)
        Cn.loc[a, b] = len(sets[a] & sets[b]) / max(len(sets[a]), 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
sns.heatmap(J, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax[0], cbar=False)
ax[0].set_title("Jaccard index\n(shared / union)")
sns.heatmap(Cn, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax[1], cbar=False)
ax[1].set_title("Fraction of the ROW file found in the COLUMN file")
plt.tight_layout(); plt.show()

**How to read the heatmap on the right.** The entry (row `PLINK`, column `ANGSD`) is the fraction of PLINK's variants that are also in the ANGSD file. If PLINK is a stricter subset, this is close to 1, whereas the transpose entry (row `ANGSD`, column `PLINK`) is small.

### 13.3 Where are the variants? Positions along the genome

Counting variants per chromosome and per window shows whether the three files sample the genome in the same way, or whether one program preferentially keeps variants in certain regions (for instance the best-covered ones).

In [ ]:
# ---- per chromosome
per = pd.DataFrame({k: S[k].groupby("contig").size() for k in names}).reindex(ANALYSED).fillna(0)
per["universe"] = U.groupby("contig").size().reindex(ANALYSED).fillna(0)
lab = [CHR_LABEL[c] for c in ANALYSED]
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
w = 0.27
for i, k in enumerate(names):
    ax[0].bar(np.arange(len(ANALYSED)) + (i - 1) * w, per[k].values, width=w, color=TOOL_COLORS[k], label=k)
ax[0].set_xticks(range(len(ANALYSED))); ax[0].set_xticklabels(lab, rotation=90, fontsize=7)
ax[0].set(xlabel="chromosome", ylabel="variants", title="Variants per chromosome"); ax[0].legend()
frac = per[names].div(per["universe"].replace(0, np.nan), axis=0)
for k in names:
    ax[1].plot(range(len(ANALYSED)), frac[k].values, "-o", ms=3, color=TOOL_COLORS[k], label=k)
ax[1].set_xticks(range(len(ANALYSED))); ax[1].set_xticklabels(lab, rotation=90, fontsize=7)
ax[1].set(xlabel="chromosome", ylabel="fraction of the universe kept", title="Is the loss uniform over chromosomes?", ylim=(0, 1.05)); ax[1].legend()
plt.tight_layout(); plt.show()

# ---- per 1-Mb window, genome-wide
WSIZE = 1_000_000
win_index = [f"{c}|{i}" for c in ANALYSED for i in range(int(fai.loc[c, "length"]) // WSIZE + 1)]
def window_counts(df, size=WSIZE):
    d = df[df.contig.isin(ANALYSED)]
    return (d.contig + "|" + (d.pos // size).astype(str)).value_counts().reindex(win_index).fillna(0)
W = pd.DataFrame({k: window_counts(S[k]) for k in names}); W["universe"] = window_counts(U)
W = W[W.universe > 0]
rho = W[names + ["universe"]].corr(method="spearman")
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.heatmap(rho, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax[0], cbar=False)
ax[0].set_title(f"Spearman correlation of variant counts\nin {len(W):,} windows of 1 Mb")
for a, (x, y) in zip(ax[1:], [("ANGSD", "PLINK"), ("ANGSD", "SNPTEST")]):
    a.scatter(W[x] + 1, W[y] + 1, s=8, alpha=0.5, color="grey"); a.set(xscale="log", yscale="log", xlabel=f"{x} variants per Mb (+1)", ylabel=f"{y} variants per Mb (+1)")
    a.plot([1, W[[x, y]].max().max() + 1], [1, W[[x, y]].max().max() + 1], "k--", lw=1); a.set_title(f"{y} vs {x}")
plt.tight_layout(); plt.show()

### 13.4 The M locus under the microscope

The decisive region is chromosome 26 around 6.73 Mb. The plot has two parts:

* **top**: the association signal of each program ($-\log_{10}P$) for the variants it tested;
* **bottom**: a **rug plot** with one tick per variant of each VCF file. Gaps in a rug are regions where that file has *no variants*, and therefore no power: if a program loses the variants inside the TE insertions (where `mm` samples have no reads), the signal can only come from the flanks.

In [ ]:
PAD = 60_000
x0, x1 = PEAK_START - PAD, PEAK_END + PAD
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1.4], "hspace": 0.08})
for k in names:
    v = res[k][(res[k].contig == M_CHROM) & res[k].pos.between(x0, x1)]
    a1.scatter(v.pos / 1e6, -np.log10(v.p.clip(lower=1e-300)), s=22, color=TOOL_COLORS[k], alpha=0.8, label=f"{k} (n={len(v)})")
a1.axvspan(PEAK_START / 1e6, PEAK_END / 1e6, color="red", alpha=0.12, label="associated window (paper)")
a1.set(ylabel=r"$-\log_{10}$ P", title=f"Association near the M locus ({M_CHROM}, {x0/1e6:.2f}-{x1/1e6:.2f} Mb)"); a1.legend(fontsize=8)
for j, k in enumerate(["universe"] + names):
    df = U if k == "universe" else S[k]
    v = df[(df.contig == M_CHROM) & df.pos.between(x0, x1)]
    a2.vlines(v.pos / 1e6, j - 0.4, j + 0.4, color=("black" if k == "universe" else TOOL_COLORS[k]), lw=1)
a2.axvspan(PEAK_START / 1e6, PEAK_END / 1e6, color="red", alpha=0.12)
a2.set_yticks(range(4)); a2.set_yticklabels(["ANGSD calls"] + names); a2.invert_yaxis()
a2.set_xlabel(f"position on {M_CHROM} (Mb)")
plt.show()

# variant density along the whole M chromosome in 100-kb windows
BIN = 100_000; L = int(fai.loc[M_CHROM, "length"])
edges = np.arange(0, L + BIN, BIN)
fig, ax = plt.subplots(figsize=(13, 3.2))
for k in ["universe"] + names:
    df = U if k == "universe" else S[k]
    h, _ = np.histogram(df.pos[df.contig == M_CHROM], bins=edges)
    ax.step(edges[:-1] / 1e6, h, where="post", color=("black" if k == "universe" else TOOL_COLORS[k]), lw=1.4, label=k if k != "universe" else "ANGSD calls")
ax.axvspan(PEAK_START / 1e6, PEAK_END / 1e6, color="red", alpha=0.25)
ax.set(xlabel=f"position on {M_CHROM} (Mb)", ylabel="variants per 100 kb", title="Variant density along the M chromosome"); ax.legend(ncol=4)
plt.tight_layout(); plt.show()

### 13.5 Do the files agree on the alleles and allele frequencies?

For variants present in two files, `REF` and `ALT` should be **identical**. Differences can occur in a way that is harmless for the test but important for combining files:

* **swapped** (`REF` of one file equals `ALT` of the other): the program relabelled the alleles, for example PLINK's choice of "A1" or SNPTEST's alleleA/alleleB. Effect sizes would change sign.
* **different**: a real inconsistency (should not happen here, all files derive from the same ANGSD calls).

The allele frequency (folded to be at most 0.5) is compared for variants shared by all three. PLINK's frequency is computed from **called genotypes only**, ANGSD's from all individuals by EM, SNPTEST's from the probabilities. Differences reflect the bias of hard calling.

In [ ]:
rows = []
for a, b in itertools.combinations(names, 2):
    m = S[a].merge(S[b], on="key", suffixes=("_a", "_b"))
    same = (m.ref_a == m.ref_b) & (m.alt_a == m.alt_b)
    swap = (m.ref_a == m.alt_b) & (m.alt_a == m.ref_b)
    rows.append(dict(pair=f"{a} vs {b}", shared_variants=len(m), identical_alleles=int(same.sum()), swapped=int(swap.sum()),
                     different=int((~same & ~swap).sum())))
display(pd.DataFrame(rows))

mm3 = res["ANGSD"][["key", "maf"]].rename(columns={"maf": "ANGSD"})
for k in ["PLINK", "SNPTEST"]:
    mm3 = mm3.merge(res[k][["key", "maf"]].rename(columns={"maf": k}), on="key")
print(f"{len(mm3):,} variants have a frequency in all three programs")
if len(mm3):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.3))
    for a, k in zip(ax, ["PLINK", "SNPTEST"]):
        a.scatter(mm3.ANGSD, mm3[k], s=6, alpha=0.4, color=TOOL_COLORS[k]); a.plot([0, 0.5], [0, 0.5], "k--", lw=1)
        a.set(xlabel="MAF, ANGSD (EM on genotype likelihoods)", ylabel=f"MAF, {k}", xlim=(0, 0.52), ylim=(0, 0.52),
              title=f"{k} vs ANGSD  (Spearman {stats.spearmanr(mm3.ANGSD, mm3[k])[0]:.2f}, mean difference {np.mean(mm3[k] - mm3.ANGSD):+.3f})")
    plt.tight_layout(); plt.show()

### 13.6 Do the programs agree on the association?

We compare the p-values of the variants tested by two programs (**shared variants only**), and then ask the practical question: *what does each program conclude?* For each program we report

* the number of tests and the **Bonferroni threshold** $0.05/n$ (a different $n$ for each file),
* the smallest p-value in the genome and where it is,
* the number of variants that pass the threshold, and
* what happens **inside the M-locus window**: the number of variants tested there, the smallest p-value, and its rank among all variants of that program.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, (x, y) in zip(ax, itertools.combinations(names, 2)):
    m = res[x][["key", "contig", "pos", "p"]].merge(res[y][["key", "p"]], on="key", suffixes=("_x", "_y"))
    lx, ly = -np.log10(m.p_x.clip(lower=1e-300)), -np.log10(m.p_y.clip(lower=1e-300))
    a.scatter(lx, ly, s=6, alpha=0.4, color="grey")
    pk = in_peak(m)
    if pk.any(): a.scatter(lx[pk], ly[pk], s=25, color="red", zorder=4, label="M window")
    top = max(lx.max(), ly.max(), 1) * 1.03
    a.plot([0, top], [0, top], "k--", lw=1)
    a.set(xlabel=f"-log10 P, {x}", ylabel=f"-log10 P, {y}", title=f"{x} vs {y}: {len(m):,} shared, Spearman {stats.spearmanr(m.p_x, m.p_y)[0]:.2f}")
    if pk.any(): a.legend()
plt.tight_layout(); plt.show()

rows = []
for k, v in res.items():
    n = len(v); thr = 0.05 / n
    best = v.loc[v.p.idxmin()]
    pk = v[in_peak(v)]
    rows.append(dict(program=k, tests=n, bonferroni_threshold=thr, lambda_GC=lambda_gc(v.p),
                     smallest_P=best.p, at=f"{CHR_LABEL[best.contig]}:{int(best.pos):,}", significant=int((v.p < thr).sum()),
                     tested_in_M_window=len(pk), smallest_P_in_M_window=(pk.p.min() if len(pk) else np.nan),
                     rank_of_that_variant=(int((v.p < pk.p.min()).sum() + 1) if len(pk) else np.nan)))
cmp_tab = pd.DataFrame(rows)
display(cmp_tab.style.format({"tests": "{:,}", "bonferroni_threshold": "{:.2e}", "lambda_GC": "{:.2f}", "smallest_P": "{:.2e}",
                              "smallest_P_in_M_window": "{:.2e}"}, na_rep="no variant"))

### 13.7 Where did the variants go? Attrition at every stage

The last view makes the *reasons* for the differences explicit. Starting from the same ANGSD call set, each program loses variants for its own reasons:

* **ANGSD:** sites for which the association test could not be computed (`-minCount`, `-minHigh`, convergence);
* **PLINK:** `--geno` (too many missing hard calls), `--maf`, `--hwe`, then tests with no valid P;
* **SNPTEST:** tests that failed, then the information filter.

In [ ]:
att = {
    "ANGSD":   [("ANGSD calls", len(U)), ("sites in -doAsso file", len(asso[ASSO_PRIMARY])), ("test succeeded", int(asso[ASSO_PRIMARY].tested.sum()))],
    "PLINK":   [("ANGSD calls", len(U))] + [(k.replace("hard-called variants (PLINK import)", "hard-call VCF imported"), v) for k, v in plink_counts.items()]
               + [("valid P", int(plink_res.p.notna().sum()))],
    "SNPTEST": [("ANGSD calls", len(U)), ("rows in output", len(snptest_all)), ("valid P", int(snptest_all.p.notna().sum())),
                (f"info >= {SNPTEST_MIN_INFO}", len(res["SNPTEST"]))],
}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for a, (k, st) in zip(ax, att.items()):
    labels = [s for s, _ in st]; vals = [v for _, v in st]
    a.barh(range(len(vals)), vals, color=TOOL_COLORS[k])
    for i, v in enumerate(vals): a.text(v, i, f" {v:,}", va="center", fontsize=8)
    a.set_yticks(range(len(vals))); a.set_yticklabels(labels, fontsize=8); a.invert_yaxis()
    a.set_xlim(0, max(vals) * 1.25); a.set_title(f"{k}: from the ANGSD calls to the VCF")
plt.tight_layout(); plt.show()

## 14. Discussion: What Did We Learn?

Use the tables and plots above to answer these questions. The answers depend on how much sequence your FASTQ subsets contain, which is why the notebook does not state numbers.

**On the variant sets (13.1 to 13.4)**

1. Which VCF has the most variants and which the fewest? Which filter, in the attrition plot, removed the most? Could a different cut-off have changed the ranking? (Try `POST_CUTOFF = 0.8`, `PLINK_GENO = 0.99`, `SNPTEST_MIN_INFO = 0.3`.)
2. Are the variants of the smaller files a **subset** of the larger ones (heatmap in 13.2), or do the files contain variants that the others lack? Why would a program contain a variant that is absent from ANGSD's calling run?
3. Are the variants *lost* by PLINK evenly distributed along the chromosomes (13.3), or concentrated in poorly covered regions? What does this mean for the **resolution** of the GWAS?
4. How many variants of each file lie in the M-locus window, and does the rug plot show gaps? Remember that `mm` individuals have no reads inside the TE insertions.

**On the association (13.5 to 13.6)**

5. Are the p-values of the three programs rank-correlated? Where do they disagree most: for the weakest or for the strongest signals?
6. Are the $\lambda_{GC}$ values close to 1? Which program is most inflated, and which choices (prior, `-minHigh`, hard-call filters, covariates) could explain it? Re-run with `N_PCS_COV = 2` and compare.
7. Does each program find the M locus? If the most significant variant is **near** but not **inside** the window, is that surprising given linkage disequilibrium and the TE insertions?
8. The Bonferroni thresholds differ between programs. How would the number of "significant" variants change if every program used the same threshold, namely the one computed from the ANGSD universe?

**On methodology**

9. At which sequencing depth would you expect PLINK and ANGSD to give nearly the same answer? (Hint: look again at Section 4.2.)
10. Which of the three programs would you choose for this dataset, and what would you check before trusting a new hit?

### 14.1 Suggested extensions

* **Other genetic models.** The phenotype is determined by a *dominant* allele. In ANGSD use `-model 2` (dominant) or `-model 3` (recessive) with `-doAsso 2`, in SNPTEST use `-frequentist 2` or `3`, in PLINK use `--model` (Fisher tests with dominant, recessive and genotypic models) or `--logistic dominant`/`recessive`. Which model gives the strongest signal at the M locus?
* **Depth as covariate.** Add `mean_depth` (log-transformed) to the covariate files.
* **Imputation.** The standard way to boost power at low coverage is to impute genotypes from the genotype likelihoods (BEAGLE, STITCH or GLIMPSE) before running PLINK or SNPTEST. The ANGSD manual includes an example that feeds BEAGLE-imputed probabilities back into `-doAsso 2`.
* **Other coverage.** Re-run the whole notebook on a different subset of BAMs (say only the 100 deepest samples) and watch how the three VCFs converge as depth increases.

## 15. Files Created

Everything is on your Drive under `association_HypMis/` next to the `bams/` folder. The cell below lists the main deliverables.

In [ ]:
deliver = {
    "ANGSD genotype likelihoods (BEAGLE)": f"{CALL_PREFIX}.beagle.gz",
    "ANGSD allele frequencies": f"{CALL_PREFIX}.mafs.gz",
    "ANGSD calls as VCF (universe)": CALLS_VCF,
    "pcangsd covariance matrix": f"{PCA_PREFIX}.cov",
    "principal components": D["pcangsd"] / "pcs.tsv",
    "ANGSD association (primary mode)": f"{D['angsd'] / ('assoc_mode' + str(ASSO_PRIMARY))}.lrt0.gz",
    "PLINK result table": assoc_file,
    "SNPTEST result table": SNPTEST_OUT,
    "VCF 1 (ANGSD)": VCF_OUT["ANGSD"], "VCF 2 (PLINK)": VCF_OUT["PLINK"], "VCF 3 (SNPTEST)": VCF_OUT["SNPTEST"],
}
rows = [(k, str(Path(v).relative_to(PROJECT_DIR)) if str(v).startswith(str(PROJECT_DIR)) else str(v),
         f"{Path(v).stat().st_size/1e6:.2f} MB" if Path(v).exists() else "MISSING") for k, v in deliver.items()]
display(pd.DataFrame(rows, columns=["content", "file (relative to the project folder)", "size"]))

# one tidy table with the three results side by side (shared variants only), handy for further work
merged = res["ANGSD"][["contig", "pos", "key", "maf", "p"]].rename(columns={"maf": "maf_ANGSD", "p": "P_ANGSD"})
for k in ["PLINK", "SNPTEST"]:
    merged = merged.merge(res[k][["key", "maf", "p"]].rename(columns={"maf": f"maf_{k}", "p": f"P_{k}"}), on="key", how="outer")
merged.to_csv(OUT / "association_results_all_programs.tsv", sep="\t", index=False)
print("Combined table written:", OUT / "association_results_all_programs.tsv", f"({len(merged):,} variants in the union)")

## 16. Summary

In this notebook we:

✅ Installed **Conda** and **ANGSD, pcangsd, PLINK, SAMtools, BCFtools/HTSlib and SNPTEST** in Google Colab.
✅ Started from the **`bams/` folder** of the QC notebook, checked every BAM (`quickcheck`, read-group `SM` tag), and verified that sequencing depth does not differ systematically between `m` and `i`.
✅ Understood, with a simulation, why genotypes cannot be hard-called at <1x and what **genotype likelihoods** and **posteriors** are.
✅ Ran a **depth pilot** to choose `-minInd` and `-setMaxDepth`.
✅ Did **probabilistic variant calling with ANGSD** (`-GL 2`, `-SNP_pval`, `-doPost 1`) and obtained `.beagle.gz`, `.mafs.gz` and a BCF/VCF.
✅ Ran **PCA with pcangsd** (admixture was intentionally left out), and checked whether the PCs relate to the morph or to depth.
✅ Tested **`m` vs `i`** with **ANGSD `-doAsso`** (modes 1, 2 and 4), **PLINK** (hard calls) and **SNPTEST** (genotype probabilities), with Manhattan plots, QQ plots and $\lambda_{GC}$.
✅ Wrote **three VCF files**, one per program, each containing the variants that program tested and its statistics in `INFO`, and **compared** them for the number of variants, their positions, alleles, allele frequencies and p-values.

### Take-home messages

* **Low-coverage data are uncertain data.** Hard calls throw away most of the information; likelihood and posterior based methods keep it. The price is a more elaborate pipeline.
* **The three programs do not test the same set of variants.** Differences in the number and position of tested variants arise from the way genotype uncertainty is handled and from program-specific filters. They change the multiple-testing burden and the resolution of the study.
* **Always look at the positive control.** The M locus on chromosome 26 tells us whether the phenotypes, BAM files, reference and coordinates are consistent, and how much signal each program recovers.
* **Check inflation (QQ, $\lambda_{GC}$) and population structure (PCA)** before interpreting any Manhattan plot.
* **Keep the sample order explicit.** ANGSD identifies individuals only by line number; the sample table, the BAM list and the phenotype file must be written together.

### References

* Korneliussen TS, Albrechtsen A, Nielsen R. (2014). *ANGSD: Analysis of Next Generation Sequencing Data.* BMC Bioinformatics 15: 356.
* Skotte L, Korneliussen TS, Albrechtsen A. (2012). *Association testing for next-generation sequencing data using score statistics.* Genetic Epidemiology 36: 430-437. (score test, `-doAsso 2`)
* Jorsboe E, Albrechtsen A. (2022). *Efficient approaches for large-scale GWAS with genotype uncertainty.* G3 (Bethesda) 12(1). (latent and dosage models, `-doAsso 4-6`)
* Meisner J, Albrechtsen A. (2018). *Inferring population structure and admixture proportions in low-depth NGS data.* Genetics 210: 719-731. (pcangsd)
* Purcell S, Neale B, Todd-Brown K, et al. (2007). *PLINK: a tool set for whole-genome association and population-based linkage analyses.* American Journal of Human Genetics 81: 559-575. Chang CC et al. (2015). *Second-generation PLINK.* GigaScience 4: 7.
* Marchini J, Howie B, Myers S, McVean G, Donnelly P. (2007). *A new multipoint method for genome-wide association studies by imputation of genotypes.* Nature Genetics 39: 906-913. Marchini J, Howie B. (2010). *Genotype imputation for genome-wide association studies.* Nature Reviews Genetics 11: 499-511. SNPTEST: <https://www.chg.ox.ac.uk/~gav/snptest/>
* Danecek P, Bonfield JK, Liddle J, et al. (2021). *Twelve years of SAMtools and BCFtools.* GigaScience 10: giab008.
* Orteu A, Kucka M, Gordon IJ, et al. (2024). *Transposable element insertions are associated with Batesian mimicry in the pantropical butterfly Hypolimnas misippus.* Mol Biol Evol 41(3): msae041. (ENA project PRJEB64669)
* Orteu A, McCarthy SA, Hornett EA, et al. (2024). *The Hypolimnas misippus genome supports a common origin of the W chromosome in Lepidoptera.* Genome Biol Evol 16(10): evae215.
* Meier JI, Salazar PA, Kucka M, et al. (2021). *Haplotype tagging reveals parallel formation of hybrid races in two butterfly species.* PNAS 118(25): e2015005118.